In [1]:
import numpy as np
import math
from math import sqrt
import cmath
import scipy
import pandas as pd
import csv
import os
from numpy import linalg
!pip install qutip
from qutip import *
from numpy.linalg import matrix_rank
from collections import Counter
import matplotlib.pyplot as plt
import scipy.stats as stats
import pandas as pd
import time

# For viewing vectors and array in matrix form
#import sympy
from sympy import Matrix

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [2]:
# Reference Paper - https://arxiv.org/pdf/2203.09610.pdf

def three_tangle_ASD(psi):
    t000 = psi[0]
    t111 = psi[7]
    tau_abc = 4*(t000**2)*(t111**2)
    return tau_abc      

def array_to_label_list(Matrix,n,k):
    '''
    Matrix refers to the density matrix which is supposed to be input in this function, 
    of whose the list is to be constructed
    
    n = Number of qubits in the system
    
    k = index reference to the three qubit entanglement class 
        k = 0 => FS => Full Separable => A-B-C => [1,0,0,0,0,0]
        k = 1 => bs2 => Bi-Separable-1 => A-BC => [0,1,0,0,0,0]
        k = 2 => BS2 => Bi-Separable-2 => AB-C => [0,0,1,0,0,0]
        k = 3 => BS3 => Bi-Separable-3 => AC-B => [0,0,0,1,0,0]
        k = 4 => W => Werner State => ABC =>      [0,0,0,0,1,0]
        k = 5 => w => Greenberg-Horne-Zeilinger State => ABC => [0,0,0,0,0,1]
        
    From 0 to 31 elements of the returned list real components of the density matrix(Re(rho_00),Re(rho_01),...) i.e, from 0 to N-1
    from 32 to 63 : Imaginary components of the density matrix i.e, (Im(rho_00),Im(rho_01),...) 
    from 64 to 69 : One hot encoded labels corresponding to the class of the 3 qubit entangled states
    '''
    N = 2**(2*n)
    Matrix_1 = Matrix.reshape((1,N))
    #data = np.zeros((2*N))
    data = [0]*(2*N)
    FS  = [1,0,0,0,0,0,0]
    bs2 = [0,1,0,0,0,0,1]
    BS2 = [0,0,1,0,0,0,2]
    BS3 = [0,0,0,1,0,0,3]
    W   = [0,0,0,0,1,0,4]
    GHZ = [0,0,0,0,0,1,5]
    
    for i in range(0,N):
        data[i] = np.real(Matrix_1[0][i])
    for j in range(N,2*N):
        data[j] = np.imag(Matrix_1[0][j-N])
        
    #data  = data.tolist()
    if k == 0:
        data.extend(FS)
    elif k == 1:
        data.extend(bs2)
    elif k == 2:
        data.extend(BS2)
    elif k == 3:
        data.extend(BS3)
    elif k == 4:
        data.extend(W)
    elif k == 5:
        data.extend(GHZ)
    else :
        return "Error : Corresponding state label doesn't exist, select from 0,1,2,3,4 or 5"
    return data

def labels(n):
    
    N=2**(2*n)
    fields = [None]*((2*N))
    list_classes_1 = ['FS','BS1','BS2','BS3','W','GHZ','CODE']
     
    k = 0 
    for i in range(2**n):
        for j in range(2**n):
            fields[k] = 'Rho_'+str(i)+'_'+str(j)+'_Real'
            k=k+1
            
    k = N
    for i in range(2**n):
        for j in range(2**n):
            fields[k] = 'Rho_'+str(i)+'_'+str(j)+'_Imag'
            k=k+1
    fields.extend(list_classes_1)
    return fields

def class_return_3q(k):
    if k == 0:
        return FS_ASD(0)
    elif k == 1:
        return BS1_ASD()
    elif k == 2:
        return BS2_ASD()
    elif k == 3:
        return BS3_ASD()
    elif k == 4:
        return W_ASD()
    elif k == 5:
        return GHZ_ASD()
    else :
        return "Class index out of [0,1,2,3,4,5]"
    
def data_generator_3q_class(num, k):
    data_list = []
    
    '''
    K = The type of entanglement class one would want to generate with this
            function
    k = 0 => FS => Full Separable => A-B-C => [1,0,0,0,0,0]
    k = 1 => bs2 => Bi-Separable-1 => A-BC => [0,1,0,0,0,0]
    k = 2 => BS2 => Bi-Separable-2 => AB-C => [0,0,1,0,0,0]
    k = 3 => BS3 => Bi-Separable-3 => AC-B => [0,0,0,1,0,0]
    k = 4 => W => W State => ABC =>      [0,0,0,0,1,0]
    k = 5 => w => Greenberg-Horne-Zeilinger State => ABC => [0,0,0,0,0,1]
    Num = No. of quantum states of the class required 
    '''
    
    for i in range(num):
        Matrix = class_return_3q(k)
        data = array_to_label_list(Matrix=Matrix,n=3,k=k)
        data_list.append(data)
    return data_list

def dim_reduce(dataframe):
    
    #index_list = [0,1,2,3,4,5,6,7]
    col_name_list = []
    col_list = []

    for i in range(0,8):
        for j in range(i,8):
            col_name = 'Rho_'+str(i)+'_'+str(j)+'_Real'
            col_name_list.append(col_name)
            col_list.append(dataframe[col_name])

    for i in range(0,8):
        for j in range(i,8):
            col_name = 'Rho_'+str(i)+'_'+str(j)+'_Imag'
            col_name_list.append(col_name)
            col_list.append(dataframe[col_name])

    col_list.append(dataframe[['FS','BS1','BS2','BS3','W','GHZ','CODE']])
    df = pd.concat(col_list, axis = 1)
    
    return df

def df_to_matrix(dataframe):
    
    all_rho_list_real = []
    all_rho_list_imag = []
    loop_length = dataframe.shape[0]
    
    for h in range(0, loop_length):
        rho_list_real = []
        for i in range(0,8):
            for j in range(0,8):
                col_name = 'Rho_'+str(i)+'_'+str(j)+'_Real'
                rho_list_real.append(dataframe.iloc[[h]][col_name].iloc[0])
        all_rho_list_real.append(rho_list_real)
        
    for h in range(0, loop_length):
        rho_list_imag = []
        for i in range(0,8):
            for j in range(0,8):
                col_name = 'Rho_'+str(i)+'_'+str(j)+'_Imag'
                rho_list_imag.append(dataframe.iloc[[h]][col_name].iloc[0])
        all_rho_list_imag.append(rho_list_imag)
        
    return all_rho_list_real, all_rho_list_imag

In [3]:
random_bs3_1 = [ 0.8047 + 0.0000*1j,  -0.0270 + 0.0108*1j,   0.0383 + 0.0196*1j,   0.0047 - 0.0164*1j,  -0.0110 - 0.0022*1j,   0.0426 + 0.0026*1j,   0.2586 - 0.0430*1j,  -0.0142 + 0.0118*1j,
  -0.0270 - 0.0108*1j,   0.0133 + 0.0000*1j,  -0.0222 - 0.0062*1j,  -0.0014 - 0.0010*1j,  -0.0233 + 0.0075*1j,  -0.0011 - 0.0008*1j,  -0.0133 + 0.0005*1j,   0.0035 + 0.0003*1j,
   0.0383 - 0.0196*1j,  -0.0222 + 0.0062*1j,   0.0407 + 0.0000*1j,   0.0022 + 0.0012*1j,   0.0372 - 0.0219*1j,   0.0017 - 0.0005*1j,   0.0172 - 0.0144*1j,  -0.0056 + 0.0012*1j,
   0.0047 + 0.0164*1j,  -0.0014 + 0.0010*1j,   0.0022 - 0.0012*1j,   0.0006 + 0.0000*1j,   0.0011 - 0.0036*1j,   0.0002 + 0.0009*1j,   0.0024 + 0.0043*1j,  -0.0006 + 0.0001*1j,
  -0.0110 + 0.0022*1j,  -0.0233 - 0.0075*1j,   0.0372 + 0.0219*1j,   0.0011 + 0.0036*1j,   0.0500 + 0.0000*1j,  -0.0012 + 0.0002*1j,   0.0058 - 0.0012*1j,  -0.0051 - 0.0029*1j,
   0.0426 - 0.0026*1j,  -0.0011 + 0.0008*1j,   0.0017 + 0.0005*1j,   0.0002 - 0.0009*1j,  -0.0012 - 0.0002*1j,   0.0023 + 0.0000*1j,   0.0134 - 0.0031*1j,  -0.0007 + 0.0007*1j,
   0.2586 + 0.0430*1j,  -0.0133 - 0.0005*1j,   0.0172 + 0.0144*1j,   0.0024 - 0.0043*1j,   0.0058 + 0.0012*1j,   0.0134 + 0.0031*1j,   0.0872 + 0.0000*1j,  -0.0060 + 0.0023*1j,
  -0.0142 - 0.0118*1j,   0.0035 - 0.0003*1j,  -0.0056 - 0.0012*1j,  -0.0006 - 0.0001*1j,  -0.0051 + 0.0029*1j,  -0.0007 - 0.0007*1j,  -0.0060 - 0.0023*1j,   0.0011 + 0.0000*1j]

random_bs3_2 = [0.7042 + 0.0000*1j,  -0.0364 + 0.0341*1j,  -0.0267 - 0.0164*1j,   0.0129 - 0.0011*1j,   0.0168 - 0.0036*1j,   0.0153 + 0.0138*1j,   0.3810 - 0.0710*1j,  -0.0257 + 0.0265*1j,
  -0.0364 - 0.0341*1j,   0.0048 + 0.0000*1j,   0.0022 + 0.0100*1j,  -0.0028 - 0.0009*1j,  -0.0022 - 0.0005*1j,   0.0010 - 0.0017*1j,  -0.0253 - 0.0114*1j,   0.0020 - 0.0014*1j,
  -0.0267 + 0.0164*1j,   0.0022 - 0.0100*1j,   0.0540 + 0.0000*1j,  -0.0056 + 0.0134*1j,  -0.0013 + 0.0080*1j,  -0.0007 - 0.0074*1j,   0.0058 + 0.0301*1j,  -0.0087 + 0.0003*1j,
   0.0129 + 0.0011*1j,  -0.0028 + 0.0009*1j,  -0.0056 - 0.0134*1j,   0.0040 + 0.0000*1j,   0.0022 - 0.0006*1j,  -0.0015 + 0.0009*1j,   0.0099 - 0.0071*1j,   0.0008 + 0.0025*1j,
   0.0168 + 0.0036*1j,  -0.0022 + 0.0005*1j,  -0.0013 - 0.0080*1j,   0.0022 + 0.0006*1j,   0.0015 + 0.0000*1j,  -0.0007 + 0.0005*1j,   0.0118 - 0.0027*1j,  -0.0003 + 0.0018*1j,
   0.0153 - 0.0138*1j,   0.0010 + 0.0017*1j,  -0.0007 + 0.0074*1j,  -0.0015 - 0.0009*1j,  -0.0007 - 0.0005*1j,   0.0016 + 0.0000*1j,   0.0044 - 0.0064*1j,  -0.0003 - 0.0001*1j,
   0.3810 + 0.0710*1j,  -0.0253 + 0.0114*1j,   0.0058 - 0.0301*1j,   0.0099 + 0.0071*1j,   0.0118 + 0.0027*1j,   0.0044 + 0.0064*1j,   0.2264 + 0.0000*1j,  -0.0191 + 0.0156*1j,
  -0.0257 - 0.0265*1j,   0.0020 + 0.0014*1j,  -0.0087 - 0.0003*1j,   0.0008 - 0.0025*1j,  -0.0003 - 0.0018*1j,  -0.0003 + 0.0001*1j,  -0.0191 - 0.0156*1j,   0.0036 + 0.0000*1j]


random_bs3_3 = [  0.2675 + 0.0000*1j,  -0.0233 + 0.0265*1j,   0.0090 - 0.0063*1j,   0.0053 + 0.0060*1j,  -0.0137 - 0.0246*1j,   0.0257 + 0.0085*1j,   0.3746 - 0.1240*1j,   0.0057 + 0.0165*1j,
  -0.0233 - 0.0265*1j,   0.0047 + 0.0000*1j,  -0.0016 - 0.0006*1j,   0.0002 - 0.0010*1j,  -0.0039 + 0.0049*1j,  -0.0010 - 0.0034*1j,  -0.0443 - 0.0249*1j,   0.0009 - 0.0024*1j,
   0.0090 + 0.0063*1j,  -0.0016 + 0.0006*1j,   0.0019 + 0.0000*1j,  -0.0003 + 0.0006*1j,   0.0030 - 0.0132*1j,   0.0001 + 0.0024*1j,   0.0095 + 0.0026*1j,   0.0019 + 0.0010*1j,
   0.0053 - 0.0060*1j,   0.0002 + 0.0010*1j,  -0.0003 - 0.0006*1j,   0.0004 + 0.0000*1j,  -0.0036 + 0.0020*1j,   0.0011 - 0.0007*1j,   0.0056 - 0.0093*1j,   0.0001 - 0.0003*1j,
  -0.0137 + 0.0246*1j,  -0.0039 - 0.0049*1j,   0.0030 + 0.0132*1j,  -0.0036 - 0.0020*1j,   0.1075 + 0.0000*1j,  -0.0155 + 0.0004*1j,  -0.0027 - 0.0123*1j,  -0.0006 + 0.0176*1j,
   0.0257 - 0.0085*1j,  -0.0010 + 0.0034*1j,   0.0001 - 0.0024*1j,   0.0011 + 0.0007*1j,  -0.0155 - 0.0004*1j,   0.0045 + 0.0000*1j,   0.0322 - 0.0169*1j,   0.0007 - 0.0009*1j,
   0.3746 + 0.1240*1j,  -0.0443 + 0.0249*1j,   0.0095 - 0.0026*1j,   0.0056 + 0.0093*1j,  -0.0027 + 0.0123*1j,   0.0322 + 0.0169*1j,   0.6092 + 0.0000*1j,  -0.0087 + 0.0272*1j,
   0.0057 - 0.0165*1j,   0.0009 + 0.0024*1j,   0.0019 - 0.0010*1j,   0.0001 + 0.0003*1j,  -0.0006 - 0.0176*1j,   0.0007 + 0.0009*1j,  -0.0087 - 0.0272*1j,   0.0042 + 0.0000*1j]


random_bs3_4 = [ 0.3573 + 0.0000*1j,  -0.0029 + 0.0073*1j,  -0.0097 + 0.0095*1j,  -0.0104 + 0.0020*1j,   0.0154 - 0.0212*1j,  -0.0113 + 0.0321*1j,   0.4030 - 0.1211*1j,   0.0258 + 0.0268*1j,
  -0.0029 - 0.0073*1j,   0.0030 + 0.0000*1j,   0.0007 + 0.0014*1j,  -0.0001 + 0.0001*1j,  -0.0162 + 0.0022*1j,   0.0022 - 0.0036*1j,   0.0015 + 0.0004*1j,  -0.0004 - 0.0020*1j,
  -0.0097 - 0.0095*1j,   0.0007 - 0.0014*1j,   0.0011 + 0.0000*1j,   0.0003 + 0.0003*1j,  -0.0025 + 0.0075*1j,  -0.0002 - 0.0018*1j,  -0.0096 - 0.0093*1j,  -0.0007 - 0.0013*1j,
  -0.0104 - 0.0020*1j,  -0.0001 - 0.0001*1j,   0.0003 - 0.0003*1j,   0.0003 + 0.0000*1j,   0.0005 - 0.0002*1j,   0.0005 - 0.0006*1j,  -0.0132 + 0.0010*1j,  -0.0005 - 0.0009*1j,
   0.0154 + 0.0212*1j,  -0.0162 - 0.0022*1j,  -0.0025 - 0.0075*1j,   0.0005 + 0.0002*1j,   0.0902 + 0.0000*1j,  -0.0137 + 0.0192*1j,  -0.0091 - 0.0296*1j,   0.0025 + 0.0104*1j,
  -0.0113 - 0.0321*1j,   0.0022 + 0.0036*1j,  -0.0002 + 0.0018*1j,   0.0005 + 0.0006*1j,  -0.0137 - 0.0192*1j,   0.0086 + 0.0000*1j,  -0.0295 - 0.0192*1j,   0.0028 - 0.0048*1j,
   0.4030 + 0.1211*1j,   0.0015 - 0.0004*1j,  -0.0096 + 0.0093*1j,  -0.0132 - 0.0010*1j,  -0.0091 + 0.0296*1j,  -0.0295 + 0.0192*1j,   0.5347 + 0.0000*1j,   0.0147 + 0.0377*1j,
   0.0258 - 0.0268*1j,  -0.0004 + 0.0020*1j,  -0.0007 + 0.0013*1j,  -0.0005 + 0.0009*1j,   0.0025 - 0.0104*1j,   0.0028 + 0.0048*1j,   0.0147 - 0.0377*1j,   0.0046 + 0.0000*1j]


random_bs3_5 = [  0.3459 + 0.0000*1j,  -0.0462 + 0.0370*1j,   0.0055 + 0.0155*1j,  -0.0073 - 0.0300*1j,   0.0108 - 0.0560*1j,   0.0181 + 0.0523*1j,   0.4133 - 0.0943*1j,  -0.0062 + 0.0370*1j,
  -0.0462 - 0.0370*1j,   0.0141 + 0.0000*1j,   0.0062 - 0.0035*1j,  -0.0022 + 0.0055*1j,  -0.0165 + 0.0140*1j,   0.0044 - 0.0129*1j,  -0.0548 - 0.0304*1j,   0.0060 - 0.0048*1j,
   0.0055 - 0.0155*1j,   0.0062 + 0.0035*1j,   0.0112 + 0.0000*1j,  -0.0017 + 0.0012*1j,  -0.0177 + 0.0115*1j,   0.0048 - 0.0060*1j,   0.0209 - 0.0150*1j,   0.0020 + 0.0014*1j,
  -0.0073 + 0.0300*1j,  -0.0022 - 0.0055*1j,  -0.0017 - 0.0012*1j,   0.0029 + 0.0000*1j,   0.0066 + 0.0037*1j,  -0.0057 + 0.0003*1j,  -0.0003 + 0.0354*1j,  -0.0030 - 0.0014*1j,
   0.0108 + 0.0560*1j,  -0.0165 - 0.0140*1j,  -0.0177 - 0.0115*1j,   0.0066 - 0.0037*1j,   0.0517 + 0.0000*1j,  -0.0195 + 0.0121*1j,   0.0059 + 0.0337*1j,  -0.0079 + 0.0004*1j,
   0.0181 - 0.0523*1j,   0.0044 + 0.0129*1j,   0.0048 + 0.0060*1j,  -0.0057 - 0.0003*1j,  -0.0195 - 0.0121*1j,   0.0135 + 0.0000*1j,   0.0085 - 0.0552*1j,   0.0060 + 0.0034*1j,
   0.4133 + 0.0943*1j,  -0.0548 + 0.0304*1j,   0.0209 + 0.0150*1j,  -0.0003 - 0.0354*1j,   0.0059 - 0.0337*1j,   0.0085 + 0.0552*1j,   0.5552 + 0.0000*1j,  -0.0161 + 0.0426*1j,
  -0.0062 - 0.0370*1j,   0.0060 + 0.0048*1j,   0.0020 - 0.0014*1j,  -0.0030 + 0.0014*1j,  -0.0079 - 0.0004*1j,   0.0060 - 0.0034*1j,  -0.0161 - 0.0426*1j,   0.0053 + 0.0000*1j]



random_ghz_1 = [   0.7297 + 0.0000*1j,  -0.0148 + 0.0350*1j,  -0.0772 - 0.0222*1j,   0.0414 + 0.0153*1j,   0.0297 + 0.0738*1j,   0.0389 - 0.0006*1j,   0.0751 - 0.0250*1j,   0.2473 + 0.0454*1j,
  -0.0148 - 0.0350*1j,   0.0248 + 0.0000*1j,   0.0231 - 0.0022*1j,   0.0098 - 0.0074*1j,  -0.0054 - 0.0028*1j,   0.0018 - 0.0020*1j,  -0.0038 - 0.0080*1j,   0.0081 - 0.0136*1j,
  -0.0772 + 0.0222*1j,   0.0231 + 0.0022*1j,   0.0894 + 0.0000*1j,  -0.0061 - 0.0135*1j,  -0.0395 + 0.0035*1j,   0.0070 + 0.0044*1j,   0.0100 + 0.0011*1j,  -0.0306 - 0.0042*1j,
   0.0414 - 0.0153*1j,   0.0098 + 0.0074*1j,  -0.0061 + 0.0135*1j,   0.0140 + 0.0000*1j,   0.0028 - 0.0059*1j,   0.0008 + 0.0008*1j,   0.0005 - 0.0027*1j,   0.0250 - 0.0016*1j,
   0.0297 - 0.0738*1j,  -0.0054 + 0.0028*1j,  -0.0395 - 0.0035*1j,   0.0028 + 0.0059*1j,   0.0264 + 0.0000*1j,  -0.0027 - 0.0070*1j,  -0.0066 - 0.0112*1j,   0.0150 - 0.0156*1j,
   0.0389 + 0.0006*1j,   0.0018 + 0.0020*1j,   0.0070 - 0.0044*1j,   0.0008 - 0.0008*1j,  -0.0027 + 0.0070*1j,   0.0038 + 0.0000*1j,   0.0065 - 0.0023*1j,   0.0118 + 0.0019*1j,
   0.0751 + 0.0250*1j,  -0.0038 + 0.0080*1j,   0.0100 - 0.0011*1j,   0.0005 + 0.0027*1j,  -0.0066 + 0.0112*1j,   0.0065 + 0.0023*1j,   0.0149 + 0.0000*1j,   0.0192 + 0.0132*1j,
   0.2473 - 0.0454*1j,   0.0081 + 0.0136*1j,  -0.0306 + 0.0042*1j,   0.0250 + 0.0016*1j,   0.0150 + 0.0156*1j,   0.0118 - 0.0019*1j,   0.0192 - 0.0132*1j,   0.0970 + 0.0000*1j]


 
random_ghz_2= [0.5883 + 0.0000*1j,   0.0033 + 0.0198*1j,  -0.0478 - 0.0079*1j,   0.0248 - 0.0182*1j,   0.0635 - 0.0633*1j,  -0.0132 + 0.0104*1j,   0.0832 - 0.0452*1j,   0.3506 + 0.0169*1j,
   0.0033 - 0.0198*1j,   0.0282 + 0.0000*1j,   0.0238 + 0.0161*1j,  -0.0021 - 0.0100*1j,  -0.0209 + 0.0007*1j,   0.0115 + 0.0090*1j,   0.0039 + 0.0033*1j,   0.0299 - 0.0087*1j,
  -0.0478 + 0.0079*1j,   0.0238 - 0.0161*1j,   0.0399 + 0.0000*1j,  -0.0046 + 0.0005*1j,  -0.0204 + 0.0223*1j,   0.0105 - 0.0047*1j,   0.0096 + 0.0044*1j,  -0.0012 + 0.0033*1j,
   0.0248 + 0.0182*1j,  -0.0021 + 0.0100*1j,  -0.0046 - 0.0005*1j,   0.0117 + 0.0000*1j,   0.0072 - 0.0048*1j,  -0.0115 + 0.0048*1j,   0.0038 - 0.0069*1j,   0.0230 + 0.0251*1j,
   0.0635 + 0.0633*1j,  -0.0209 - 0.0007*1j,  -0.0204 - 0.0223*1j,   0.0072 + 0.0048*1j,   0.0292 + 0.0000*1j,  -0.0113 - 0.0045*1j,   0.0085 - 0.0045*1j,   0.0230 + 0.0319*1j,
  -0.0132 - 0.0104*1j,   0.0115 - 0.0090*1j,   0.0105 + 0.0047*1j,  -0.0115 - 0.0048*1j,  -0.0113 + 0.0045*1j,   0.0153 + 0.0000*1j,  -0.0020 + 0.0093*1j,  -0.0059 - 0.0212*1j,
   0.0832 + 0.0452*1j,   0.0039 - 0.0033*1j,   0.0096 - 0.0044*1j,   0.0038 + 0.0069*1j,   0.0085 + 0.0045*1j,  -0.0020 - 0.0093*1j,   0.0290 + 0.0000*1j,   0.0500 + 0.0387*1j,
   0.3506 - 0.0169*1j,   0.0299 + 0.0087*1j,  -0.0012 - 0.0033*1j,   0.0230 - 0.0251*1j,   0.0230 - 0.0319*1j,  -0.0059 + 0.0212*1j,   0.0500 - 0.0387*1j,   0.2585 + 0.0000*1j] 
    
random_ghz_3 = [0.6446 + 0.0000*1j,  -0.0223 + 0.0107*1j,  -0.0706 - 0.0154*1j,  -0.0445 + 0.0186*1j,   0.0356 - 0.0522*1j,   0.0021 - 0.0121*1j,   0.1070 + 0.0119*1j,   0.3289 - 0.0039*1j,
  -0.0223 - 0.0107*1j,   0.0386 + 0.0000*1j,   0.0126 - 0.0041*1j,   0.0090 - 0.0069*1j,  -0.0085 + 0.0058*1j,  -0.0102 - 0.0083*1j,  -0.0140 + 0.0037*1j,  -0.0098 + 0.0122*1j,
  -0.0706 + 0.0154*1j,   0.0126 + 0.0041*1j,   0.0295 + 0.0000*1j,   0.0085 - 0.0052*1j,  -0.0189 + 0.0111*1j,   0.0025 + 0.0065*1j,  -0.0168 + 0.0085*1j,  -0.0304 + 0.0112*1j,
  -0.0445 - 0.0186*1j,   0.0090 + 0.0069*1j,   0.0085 + 0.0052*1j,   0.0160 + 0.0000*1j,  -0.0095 + 0.0031*1j,  -0.0087 - 0.0056*1j,  -0.0047 - 0.0100*1j,  -0.0393 - 0.0074*1j,
   0.0356 + 0.0522*1j,  -0.0085 - 0.0058*1j,  -0.0189 - 0.0111*1j,  -0.0095 - 0.0031*1j,   0.0205 + 0.0000*1j,   0.0023 - 0.0037*1j,   0.0075 + 0.0046*1j,   0.0169 + 0.0259*1j,
   0.0021 + 0.0121*1j,  -0.0102 + 0.0083*1j,   0.0025 - 0.0065*1j,  -0.0087 + 0.0056*1j,   0.0023 + 0.0037*1j,   0.0177 + 0.0000*1j,   0.0025 + 0.0068*1j,   0.0092 - 0.0061*1j,
   0.1070 - 0.0119*1j,  -0.0140 - 0.0037*1j,  -0.0168 - 0.0085*1j,  -0.0047 + 0.0100*1j,   0.0075 - 0.0046*1j,   0.0025 - 0.0068*1j,   0.0319 + 0.0000*1j,   0.0479 - 0.0246*1j,
   0.3289 + 0.0039*1j,  -0.0098 - 0.0122*1j,  -0.0304 - 0.0112*1j,  -0.0393 + 0.0074*1j,   0.0169 - 0.0259*1j,   0.0092 + 0.0061*1j,   0.0479 + 0.0246*1j,   0.2012 + 0.0000*1j]

random_ghz_4 = [0.1990 + 0.0000*1j,   0.0018 - 0.0061*1j,  -0.0023 - 0.0017*1j,   0.0234 - 0.0076*1j,   0.0005 - 0.0212*1j,   0.0011 + 0.0152*1j,   0.0584 - 0.0113*1j,   0.2567 + 0.0640*1j,
   0.0018 + 0.0061*1j,   0.0080 + 0.0000*1j,   0.0099 + 0.0083*1j,   0.0071 - 0.0013*1j,  -0.0180 - 0.0070*1j,  -0.0073 - 0.0055*1j,   0.0051 - 0.0096*1j,  -0.0179 + 0.0143*1j,
  -0.0023 + 0.0017*1j,   0.0099 - 0.0083*1j,   0.0299 + 0.0000*1j,   0.0037 - 0.0157*1j,  -0.0515 + 0.0198*1j,   0.0009 + 0.0131*1j,  -0.0114 - 0.0024*1j,  -0.0184 + 0.0361*1j,
   0.0234 + 0.0076*1j,   0.0071 + 0.0013*1j,   0.0037 + 0.0157*1j,   0.0161 + 0.0000*1j,  -0.0122 - 0.0306*1j,  -0.0209 + 0.0037*1j,   0.0054 - 0.0219*1j,  -0.0047 + 0.0164*1j,
   0.0005 + 0.0212*1j,  -0.0180 + 0.0070*1j,  -0.0515 - 0.0198*1j,  -0.0122 + 0.0306*1j,   0.1140 + 0.0000*1j,  -0.0085 - 0.0398*1j,   0.0263 - 0.0060*1j,   0.0269 - 0.0199*1j,
   0.0011 - 0.0152*1j,  -0.0073 + 0.0055*1j,   0.0009 - 0.0131*1j,  -0.0209 - 0.0037*1j,  -0.0085 + 0.0398*1j,   0.0636 + 0.0000*1j,   0.0180 + 0.0521*1j,   0.0433 - 0.0245*1j,
   0.0584 + 0.0113*1j,   0.0051 + 0.0096*1j,  -0.0114 + 0.0024*1j,   0.0054 + 0.0219*1j,   0.0263 + 0.0060*1j,   0.0180 - 0.0521*1j,   0.1042 + 0.0000*1j,   0.0531 - 0.0402*1j,
   0.2567 - 0.0640*1j,  -0.0179 - 0.0143*1j,  -0.0184 - 0.0361*1j,  -0.0047 - 0.0164*1j,   0.0269 + 0.0199*1j,   0.0433 + 0.0245*1j,   0.0531 + 0.0402*1j,   0.4652 + 0.0000*1j]
    
random_ghz_5 = [0.3450 + 0.0000*1j,   0.0155 + 0.0463*1j,  -0.0247 - 0.0289*1j,  -0.0041 + 0.0218*1j,   0.0495 - 0.0071*1j,   0.0510 - 0.0278*1j,   0.0479 - 0.0357*1j,   0.2265 + 0.0523*1j,
   0.0155 - 0.0463*1j,   0.1018 + 0.0000*1j,   0.0202 - 0.0209*1j,   0.0053 - 0.0018*1j,   0.0081 - 0.0007*1j,  -0.0121 - 0.0318*1j,  -0.0875 - 0.0548*1j,   0.0101 - 0.0143*1j,
  -0.0247 + 0.0289*1j,   0.0202 + 0.0209*1j,   0.0612 + 0.0000*1j,  -0.0181 - 0.0108*1j,  -0.0257 + 0.0132*1j,   0.0552 + 0.0157*1j,  -0.0362 - 0.0205*1j,  -0.0228 + 0.0187*1j,
  -0.0041 - 0.0218*1j,   0.0053 + 0.0018*1j,  -0.0181 + 0.0108*1j,   0.0109 + 0.0000*1j,   0.0077 - 0.0096*1j,  -0.0256 - 0.0029*1j,   0.0084 - 0.0158*1j,  -0.0047 - 0.0170*1j,
   0.0495 + 0.0071*1j,   0.0081 + 0.0007*1j,  -0.0257 - 0.0132*1j,   0.0077 + 0.0096*1j,   0.0217 + 0.0000*1j,  -0.0159 - 0.0206*1j,   0.0178 - 0.0010*1j,   0.0288 + 0.0062*1j,
   0.0510 + 0.0278*1j,  -0.0121 + 0.0318*1j,   0.0552 - 0.0157*1j,  -0.0256 + 0.0029*1j,  -0.0159 + 0.0206*1j,   0.1032 + 0.0000*1j,   0.0251 + 0.0083*1j,   0.0306 - 0.0004*1j,
   0.0479 + 0.0357*1j,  -0.0875 + 0.0548*1j,  -0.0362 + 0.0205*1j,   0.0084 + 0.0158*1j,   0.0178 + 0.0010*1j,   0.0251 - 0.0083*1j,   0.1585 + 0.0000*1j,   0.0097 - 0.0199*1j,
   0.2265 - 0.0523*1j,   0.0101 + 0.0143*1j,  -0.0228 - 0.0187*1j,  -0.0047 + 0.0170*1j,   0.0288 - 0.0062*1j,   0.0306 + 0.0004*1j,   0.0097 + 0.0199*1j,   0.1977 + 0.0000*1j]
 
random_ghz_6 = [ 0.5729 + 0.0000*1j,   0.0126 + 0.0126*1j,  -0.0496 - 0.0141*1j,  -0.0572 - 0.0087*1j,  -0.0189 + 0.0464*1j,  -0.0434 - 0.0139*1j,   0.0550 + 0.0034*1j,   0.3464 + 0.0330*1j,
   0.0126 - 0.0126*1j,   0.0237 + 0.0000*1j,  -0.0009 + 0.0002*1j,   0.0003 - 0.0083*1j,  -0.0108 + 0.0037*1j,  -0.0049 + 0.0000*1j,  -0.0021 + 0.0040*1j,   0.0287 + 0.0060*1j,
  -0.0496 + 0.0141*1j,  -0.0009 - 0.0002*1j,   0.0385 + 0.0000*1j,  -0.0160 + 0.0070*1j,  -0.0216 + 0.0019*1j,   0.0237 - 0.0073*1j,  -0.0119 + 0.0266*1j,  -0.0180 - 0.0243*1j,
  -0.0572 + 0.0087*1j,   0.0003 + 0.0083*1j,  -0.0160 - 0.0070*1j,   0.0252 + 0.0000*1j,   0.0147 - 0.0086*1j,  -0.0096 - 0.0006*1j,   0.0023 - 0.0151*1j,  -0.0539 + 0.0278*1j,
  -0.0189 - 0.0464*1j,  -0.0108 - 0.0037*1j,  -0.0216 - 0.0019*1j,   0.0147 + 0.0086*1j,   0.0256 + 0.0000*1j,  -0.0120 + 0.0055*1j,   0.0101 - 0.0219*1j,  -0.0319 - 0.0203*1j,
  -0.0434 + 0.0139*1j,  -0.0049 - 0.0000*1j,   0.0237 + 0.0073*1j,  -0.0096 + 0.0006*1j,  -0.0120 - 0.0055*1j,   0.0172 + 0.0000*1j,  -0.0134 + 0.0135*1j,  -0.0165 - 0.0106*1j,
   0.0550 - 0.0034*1j,  -0.0021 - 0.0040*1j,  -0.0119 - 0.0266*1j,   0.0023 + 0.0151*1j,   0.0101 + 0.0219*1j,  -0.0134 - 0.0135*1j,   0.0276 + 0.0000*1j,   0.0075 - 0.0088*1j,
   0.3464 - 0.0330*1j,   0.0287 - 0.0060*1j,  -0.0180 + 0.0243*1j,  -0.0539 - 0.0278*1j,  -0.0319 + 0.0203*1j,  -0.0165 + 0.0106*1j,   0.0075 + 0.0088*1j,   0.2693 + 0.0000*1j]    
    
random_sep_2 = [  0.0205 + 0.0000*1j,  -0.0010 + 0.0019*1j,   0.0040 - 0.0086*1j,   0.0010 + 0.0001*1j,   0.0891 - 0.0067*1j,  -0.0031 + 0.0046*1j,   0.0629 - 0.0141*1j,  -0.0017 + 0.0057*1j,
  -0.0010 - 0.0019*1j,   0.0019 + 0.0000*1j,  -0.0000 - 0.0026*1j,   0.0011 - 0.0004*1j,  -0.0046 + 0.0140*1j,   0.0013 + 0.0001*1j,  -0.0050 + 0.0069*1j,   0.0013 - 0.0012*1j,
   0.0040 + 0.0086*1j,  -0.0000 + 0.0026*1j,   0.0089 + 0.0000*1j,   0.0013 + 0.0020*1j,  -0.0137 + 0.0484*1j,  -0.0021 + 0.0008*1j,  -0.0010 + 0.0291*1j,  -0.0007 + 0.0008*1j,
   0.0010 - 0.0001*1j,   0.0011 + 0.0004*1j,   0.0013 - 0.0020*1j,   0.0009 + 0.0000*1j,  -0.0001 + 0.0138*1j,   0.0004 + 0.0004*1j,  -0.0000 + 0.0069*1j,   0.0006 - 0.0003*1j,
   0.0891 + 0.0067*1j,  -0.0046 - 0.0140*1j,  -0.0137 - 0.0484*1j,  -0.0001 - 0.0138*1j,   0.6738 + 0.0000*1j,  -0.0149 + 0.0092*1j,   0.4347 - 0.0299*1j,  -0.0233 + 0.0155*1j,
  -0.0031 - 0.0046*1j,   0.0013 - 0.0001*1j,  -0.0021 - 0.0008*1j,   0.0004 - 0.0004*1j,  -0.0149 - 0.0092*1j,   0.0018 + 0.0000*1j,  -0.0130 - 0.0067*1j,   0.0018 - 0.0009*1j,
   0.0629 + 0.0141*1j,  -0.0050 - 0.0069*1j,  -0.0010 - 0.0291*1j,  -0.0000 - 0.0069*1j,   0.4347 + 0.0299*1j,  -0.0130 + 0.0067*1j,   0.2895 + 0.0000*1j,  -0.0172 + 0.0121*1j,
  -0.0017 - 0.0057*1j,   0.0013 + 0.0012*1j,  -0.0007 - 0.0008*1j,   0.0006 + 0.0003*1j,  -0.0233 - 0.0155*1j,   0.0018 + 0.0009*1j,  -0.0172 - 0.0121*1j,   0.0027 + 0.0000*1j]
        
random_sep_3= [ 0.1445 + 0.0000*1j,  -0.0043 + 0.0027*1j,  -0.0273 - 0.0217*1j,  -0.0016 - 0.0016*1j,   0.3209 - 0.0138*1j,  -0.0075 + 0.0064*1j,   0.0383 - 0.0294*1j,  -0.0092 + 0.0051*1j,
  -0.0043 - 0.0027*1j,   0.0009 + 0.0000*1j,  -0.0002 + 0.0001*1j,  -0.0003 - 0.0006*1j,  -0.0108 - 0.0018*1j,   0.0012 + 0.0009*1j,  -0.0040 - 0.0053*1j,   0.0022 - 0.0006*1j,
  -0.0273 + 0.0217*1j,  -0.0002 - 0.0001*1j,   0.0117 + 0.0000*1j,   0.0017 + 0.0001*1j,  -0.0660 + 0.0448*1j,  -0.0013 - 0.0029*1j,   0.0126 + 0.0131*1j,   0.0006 + 0.0010*1j,
  -0.0016 + 0.0016*1j,  -0.0003 + 0.0006*1j,   0.0017 - 0.0001*1j,   0.0007 + 0.0000*1j,  -0.0061 + 0.0012*1j,  -0.0013 + 0.0003*1j,   0.0052 + 0.0013*1j,  -0.0002 + 0.0017*1j,
   0.3209 + 0.0138*1j,  -0.0108 + 0.0018*1j,  -0.0660 - 0.0448*1j,  -0.0061 - 0.0012*1j,   0.7429 + 0.0000*1j,  -0.0130 + 0.0101*1j,   0.0501 - 0.0371*1j,  -0.0274 + 0.0017*1j,
  -0.0075 - 0.0064*1j,   0.0012 - 0.0009*1j,  -0.0013 + 0.0029*1j,  -0.0013 - 0.0003*1j,  -0.0130 - 0.0101*1j,   0.0042 + 0.0000*1j,  -0.0122 + 0.0014*1j,   0.0020 - 0.0030*1j,
   0.0383 + 0.0294*1j,  -0.0040 + 0.0053*1j,   0.0126 - 0.0131*1j,   0.0052 - 0.0013*1j,   0.0501 + 0.0371*1j,  -0.0122 - 0.0014*1j,   0.0891 + 0.0000*1j,  -0.0037 + 0.0156*1j,
  -0.0092 - 0.0051*1j,   0.0022 + 0.0006*1j,   0.0006 - 0.0010*1j,  -0.0002 - 0.0017*1j,  -0.0274 - 0.0017*1j,   0.0020 + 0.0030*1j,  -0.0037 - 0.0156*1j,   0.0060 + 0.0000*1j]
  
    
random_sep_4 = [  0.0274 + 0.0000*1j,   0.0060 + 0.0069*1j,  -0.0023 + 0.0098*1j,  -0.0032 + 0.0065*1j,   0.0320 - 0.0972*1j,   0.0290 - 0.0435*1j,   0.0211 - 0.0201*1j,   0.0138 - 0.0006*1j,
   0.0060 - 0.0069*1j,   0.0031 + 0.0000*1j,   0.0023 + 0.0024*1j,   0.0011 + 0.0022*1j,  -0.0170 - 0.0270*1j,  -0.0047 - 0.0155*1j,  -0.0002 - 0.0106*1j,   0.0033 - 0.0039*1j,
  -0.0023 - 0.0098*1j,   0.0023 - 0.0024*1j,   0.0143 + 0.0000*1j,   0.0055 + 0.0023*1j,  -0.0627 + 0.0455*1j,  -0.0398 + 0.0142*1j,   0.0109 - 0.0146*1j,   0.0114 - 0.0025*1j,
  -0.0032 - 0.0065*1j,   0.0011 - 0.0022*1j,   0.0055 - 0.0023*1j,   0.0030 + 0.0000*1j,  -0.0259 + 0.0211*1j,  -0.0163 + 0.0073*1j,  -0.0034 - 0.0082*1j,   0.0021 - 0.0046*1j,
   0.0320 + 0.0972*1j,  -0.0170 + 0.0270*1j,  -0.0627 - 0.0455*1j,  -0.0259 - 0.0211*1j,   0.6684 + 0.0000*1j,   0.3368 + 0.1028*1j,   0.0084 - 0.0193*1j,  -0.0012 - 0.0162*1j,
   0.0290 + 0.0435*1j,  -0.0047 + 0.0155*1j,  -0.0398 - 0.0142*1j,  -0.0163 - 0.0073*1j,   0.3368 - 0.1028*1j,   0.1859 + 0.0000*1j,  -0.0037 - 0.0090*1j,  -0.0059 - 0.0088*1j,
   0.0211 + 0.0201*1j,  -0.0002 + 0.0106*1j,   0.0109 + 0.0146*1j,  -0.0034 + 0.0082*1j,   0.0084 + 0.0193*1j,  -0.0037 + 0.0090*1j,   0.0752 + 0.0000*1j,   0.0329 + 0.0246*1j,
   0.0138 + 0.0006*1j,   0.0033 + 0.0039*1j,   0.0114 + 0.0025*1j,   0.0021 + 0.0046*1j,  -0.0012 + 0.0162*1j,  -0.0059 + 0.0088*1j,   0.0329 - 0.0246*1j,   0.0228 + 0.0000*1j]


random_sep_5 = [ 0.0094 + 0.0000*1j,   0.0050 + 0.0007*1j,  -0.0016 + 0.0033*1j,  -0.0004 + 0.0025*1j,   0.0046 - 0.0264*1j,   0.0058 - 0.0356*1j,   0.0136 - 0.0115*1j,   0.0140 - 0.0052*1j,
   0.0050 - 0.0007*1j,   0.0036 + 0.0000*1j,  -0.0003 + 0.0012*1j,  -0.0007 + 0.0008*1j,   0.0038 - 0.0130*1j,   0.0108 - 0.0197*1j,   0.0078 - 0.0081*1j,   0.0086 - 0.0064*1j,
  -0.0016 - 0.0033*1j,  -0.0003 - 0.0012*1j,   0.0056 + 0.0000*1j,   0.0047 + 0.0012*1j,  -0.0259 + 0.0149*1j,  -0.0471 + 0.0112*1j,   0.0015 - 0.0003*1j,   0.0032 + 0.0019*1j,
  -0.0004 - 0.0025*1j,  -0.0007 - 0.0008*1j,   0.0047 - 0.0012*1j,   0.0068 + 0.0000*1j,  -0.0221 + 0.0201*1j,  -0.0583 + 0.0308*1j,   0.0038 - 0.0039*1j,   0.0072 + 0.0026*1j,
   0.0046 + 0.0264*1j,   0.0038 + 0.0130*1j,  -0.0259 - 0.0149*1j,  -0.0221 - 0.0201*1j,   0.1877 + 0.0000*1j,   0.3173 + 0.0787*1j,   0.0166 + 0.0017*1j,   0.0149 - 0.0149*1j,
   0.0058 + 0.0356*1j,   0.0108 + 0.0197*1j,  -0.0471 - 0.0112*1j,  -0.0583 - 0.0308*1j,   0.3173 - 0.0787*1j,   0.6854 + 0.0000*1j,  -0.0046 + 0.0150*1j,  -0.0073 - 0.0454*1j,
   0.0136 + 0.0115*1j,   0.0078 + 0.0081*1j,   0.0015 + 0.0003*1j,   0.0038 + 0.0039*1j,   0.0166 - 0.0017*1j,  -0.0046 - 0.0150*1j,   0.0506 + 0.0000*1j,   0.0440 + 0.0159*1j,
   0.0140 + 0.0052*1j,   0.0086 + 0.0064*1j,   0.0032 - 0.0019*1j,   0.0072 - 0.0026*1j,   0.0149 + 0.0149*1j,  -0.0073 + 0.0454*1j,   0.0440 - 0.0159*1j,   0.0509 + 0.0000*1j]

    
    
sep_output = [0.0025 + 0.0000*1j,  -0.0018 + 0.0013*1j,  -0.0004 - 0.0025*1j,   0.0026 + 0.0005*1j,   0.0019 - 0.0042*1j,   0.0007 + 0.0049*1j,  -0.0054 - 0.0006*1j,   0.0165 - 0.0024*1j,
  -0.0018 - 0.0013*1j,   0.0034 + 0.0000*1j,  -0.0011 + 0.0011*1j,  -0.0032 - 0.0035*1j,  -0.0033 + 0.0021*1j,   0.0009 - 0.0047*1j,   0.0028 + 0.0011*1j,  -0.0341 - 0.0309*1j,
  -0.0004 + 0.0025*1j,  -0.0011 - 0.0011*1j,   0.0060 + 0.0000*1j,   0.0027 + 0.0002*1j,   0.0007 + 0.0037*1j,   0.0001 - 0.0045*1j,   0.0040 - 0.0127*1j,   0.0281 + 0.0180*1j,
   0.0026 - 0.0005*1j,  -0.0032 + 0.0035*1j,   0.0027 - 0.0002*1j,   0.0093 + 0.0000*1j,  -0.0023 - 0.0051*1j,   0.0094 + 0.0034*1j,   0.0024 - 0.0035*1j,   0.0727 + 0.0105*1j,
   0.0019 + 0.0042*1j,  -0.0033 - 0.0021*1j,   0.0007 - 0.0037*1j,  -0.0023 + 0.0051*1j,   0.0119 + 0.0000*1j,  -0.0138 + 0.0071*1j,  -0.0072 - 0.0026*1j,   0.0026 + 0.0062*1j,
   0.0007 - 0.0049*1j,   0.0009 + 0.0047*1j,   0.0001 + 0.0045*1j,   0.0094 - 0.0034*1j,  -0.0138 - 0.0071*1j,   0.0227 + 0.0000*1j,   0.0103 + 0.0025*1j,   0.0345 + 0.0083*1j,
  -0.0054 + 0.0006*1j,   0.0028 - 0.0011*1j,   0.0040 + 0.0127*1j,   0.0024 + 0.0035*1j,  -0.0072 + 0.0026*1j,   0.0103 - 0.0025*1j,   0.0331 + 0.0000*1j,  -0.0135 + 0.0259*1j,
   0.0165 + 0.0024*1j,  -0.0341 + 0.0309*1j,   0.0281 - 0.0180*1j,   0.0727 - 0.0105*1j,   0.0026 - 0.0062*1j,   0.0345 - 0.0083*1j,  -0.0135 - 0.0259*1j,   0.9111 + 0.0000*1j]


W_output = [0.2921 + 0.0000*1j,   0.0150 + 0.0168*1j,   0.0255 - 0.0057*1j,  -0.0139 + 0.0283*1j,   0.2764 - 0.0827*1j,   0.2190 + 0.0216*1j,   0.1929 - 0.0887*1j,   0.0005 + 0.0200*1j,
   0.0150 - 0.0168*1j,   0.0085 + 0.0000*1j,   0.0011 + 0.0017*1j,   0.0038 + 0.0014*1j,   0.0105 - 0.0071*1j,   0.0161 - 0.0010*1j,  -0.0005 - 0.0003*1j,   0.0041 - 0.0000*1j,
   0.0255 + 0.0057*1j,   0.0011 - 0.0017*1j,   0.0041 + 0.0000*1j,  -0.0022 + 0.0007*1j,   0.0325 - 0.0022*1j,   0.0242 + 0.0044*1j,   0.0264 - 0.0010*1j,  -0.0008 + 0.0002*1j,
  -0.0139 - 0.0283*1j,   0.0038 - 0.0014*1j,  -0.0022 - 0.0007*1j,   0.0047 + 0.0000*1j,  -0.0223 - 0.0172*1j,  -0.0081 - 0.0173*1j,  -0.0219 - 0.0086*1j,   0.0033 - 0.0010*1j,
   0.2764 + 0.0827*1j,   0.0105 + 0.0071*1j,   0.0325 + 0.0022*1j,  -0.0223 + 0.0172*1j,   0.3104 + 0.0000*1j,   0.2220 + 0.0770*1j,   0.2365 - 0.0167*1j,  -0.0067 + 0.0131*1j,
   0.2190 - 0.0216*1j,   0.0161 + 0.0010*1j,   0.0242 - 0.0044*1j,  -0.0081 + 0.0173*1j,   0.2220 - 0.0770*1j,   0.1841 + 0.0000*1j,   0.1591 - 0.0643*1j,   0.0019 + 0.0098*1j,
   0.1929 + 0.0887*1j,  -0.0005 + 0.0003*1j,   0.0264 + 0.0010*1j,  -0.0219 + 0.0086*1j,   0.2365 + 0.0167*1j,   0.1591 + 0.0643*1j,   0.1932 + 0.0000*1j,  -0.0103 + 0.0074*1j,
   0.0005 - 0.0200*1j,   0.0041 + 0.0000*1j,  -0.0008 - 0.0002*1j,   0.0033 + 0.0010*1j,  -0.0067 - 0.0131*1j,   0.0019 - 0.0098*1j,  -0.0103 - 0.0074*1j,   0.0028 + 0.0000*1j]

w1 = [  0.0968 + 0.0000*1j, -0.0279 - 0.0043*1j, -0.0307 + 0.0323*1j,  0.0293 - 0.0001*1j, -0.0274 - 0.0093*1j,  0.1234 + 0.0174*1j,  0.1584 - 0.0533*1j, -0.0332 - 0.0221*1j,
  -0.0279 + 0.0043*1j,  0.0305 + 0.0000*1j, -0.0041 - 0.0107*1j, -0.0133 - 0.0024*1j,  0.0076 - 0.0299*1j, -0.0656 - 0.0279*1j, -0.0627 - 0.0136*1j,  0.0290 + 0.0090*1j,
  -0.0307 - 0.0323*1j, -0.0041 + 0.0107*1j,  0.0282 + 0.0000*1j, -0.0070 - 0.0109*1j,  0.0062 + 0.0336*1j, -0.0260 - 0.0219*1j, -0.0588 - 0.0151*1j, -0.0052 + 0.0229*1j,
   0.0293 + 0.0001*1j, -0.0133 + 0.0024*1j, -0.0070 + 0.0109*1j,  0.0158 + 0.0000*1j, -0.0123 + 0.0038*1j,  0.0311 - 0.0079*1j,  0.0542 - 0.0133*1j, -0.0269 - 0.0030*1j,
  -0.0274 + 0.0093*1j,  0.0076 + 0.0299*1j,  0.0062 - 0.0336*1j, -0.0123 - 0.0038*1j,  0.0693 + 0.0000*1j,  0.0349 - 0.0078*1j,  0.0186 + 0.0073*1j,  0.0267 + 0.0261*1j,
   0.1234 - 0.0174*1j, -0.0656 + 0.0279*1j, -0.0260 + 0.0219*1j,  0.0311 + 0.0079*1j,  0.0349 + 0.0078*1j,  0.3312 + 0.0000*1j,  0.2818 - 0.0787*1j, -0.0401 - 0.0426*1j,
   0.1584 + 0.0533*1j, -0.0627 + 0.0136*1j, -0.0588 + 0.0151*1j,  0.0542 + 0.0133*1j,  0.0186 - 0.0073*1j,  0.2818 + 0.0787*1j,  0.3673 + 0.0000*1j, -0.0562 - 0.0337*1j,
  -0.0332 + 0.0221*1j,  0.0290 - 0.0090*1j, -0.0052 - 0.0229*1j, -0.0269 + 0.0030*1j,  0.0267 - 0.0261*1j, -0.0401 + 0.0426*1j, -0.0562 + 0.0337*1j,  0.0609 + 0.0000*1j, ]

w2 =   [ 0.5661 + 0.0000*1j,  0.0217 + 0.0013*1j,  0.0733 - 0.0336*1j, -0.0109 - 0.0782*1j,  0.0190 + 0.0306*1j,  0.1375 + 0.0080*1j,  0.2367 - 0.0446*1j, -0.0167 - 0.0120*1j,
   0.0217 - 0.0013*1j,  0.0676 + 0.0000*1j,  0.0446 - 0.0217*1j,  0.0042 + 0.0255*1j,  0.0268 - 0.0095*1j,  0.0219 + 0.0036*1j,  0.0179 - 0.0037*1j,  0.0186 + 0.0097*1j,
   0.0733 + 0.0336*1j,  0.0446 + 0.0217*1j,  0.1220 + 0.0000*1j, -0.0552 + 0.0027*1j,  0.0528 - 0.0045*1j,  0.0191 + 0.0218*1j,  0.0378 + 0.0171*1j, -0.0063 - 0.0029*1j,
  -0.0109 + 0.0782*1j,  0.0042 - 0.0255*1j, -0.0552 - 0.0027*1j,  0.0596 + 0.0000*1j, -0.0285 - 0.0000*1j,  0.0036 + 0.0078*1j,  0.0021 + 0.0248*1j,  0.0180 - 0.0022*1j,
   0.0190 - 0.0306*1j,  0.0268 + 0.0095*1j,  0.0528 + 0.0045*1j, -0.0285 + 0.0000*1j,  0.0292 + 0.0000*1j,  0.0070 - 0.0020*1j,  0.0076 - 0.0107*1j,  0.0006 + 0.0003*1j,
   0.1375 - 0.0080*1j,  0.0219 - 0.0036*1j,  0.0191 - 0.0218*1j,  0.0036 - 0.0078*1j,  0.0070 + 0.0020*1j,  0.0389 + 0.0000*1j,  0.0596 - 0.0156*1j,  0.0016 + 0.0010*1j,
   0.2367 + 0.0446*1j,  0.0179 + 0.0037*1j,  0.0378 - 0.0171*1j,  0.0021 - 0.0248*1j,  0.0076 + 0.0107*1j,  0.0596 + 0.0156*1j,  0.1043 + 0.0000*1j, -0.0045 - 0.0031*1j,
  -0.0167 + 0.0120*1j,  0.0186 - 0.0097*1j, -0.0063 + 0.0029*1j,  0.0180 + 0.0022*1j,  0.0006 - 0.0003*1j,  0.0016 - 0.0010*1j, -0.0045 + 0.0031*1j,  0.0124 + 0.0000*1j, ] 

w3 =  [  0.7222 + 0.0000*1j, -0.0580 + 0.0685*1j, -0.0092 + 0.0385*1j, -0.0199 - 0.0060*1j, -0.0184 + 0.0268*1j,  0.2384 + 0.0210*1j,  0.2489 - 0.0518*1j, -0.0517 + 0.0112*1j,
  -0.0580 - 0.0685*1j,  0.0466 + 0.0000*1j, -0.0021 + 0.0083*1j, -0.0066 - 0.0009*1j, -0.0020 - 0.0150*1j, -0.0183 - 0.0110*1j, -0.0352 - 0.0213*1j,  0.0165 + 0.0033*1j,
  -0.0092 - 0.0385*1j, -0.0021 - 0.0083*1j,  0.0256 + 0.0000*1j, -0.0056 + 0.0088*1j,  0.0032 - 0.0003*1j, -0.0003 - 0.0144*1j, -0.0093 - 0.0020*1j,  0.0008 + 0.0003*1j,
  -0.0199 + 0.0060*1j, -0.0066 + 0.0009*1j, -0.0056 - 0.0088*1j,  0.0056 + 0.0000*1j,  0.0000 + 0.0022*1j, -0.0067 - 0.0011*1j, -0.0008 + 0.0019*1j, -0.0014 - 0.0003*1j,
  -0.0184 - 0.0268*1j, -0.0020 + 0.0150*1j,  0.0032 + 0.0003*1j,  0.0000 - 0.0022*1j,  0.0112 + 0.0000*1j, -0.0114 - 0.0127*1j, -0.0090 - 0.0114*1j,  0.0003 + 0.0071*1j,
   0.2384 - 0.0210*1j, -0.0183 + 0.0110*1j, -0.0003 + 0.0144*1j, -0.0067 + 0.0011*1j, -0.0114 + 0.0127*1j,  0.0846 + 0.0000*1j,  0.0811 - 0.0212*1j, -0.0176 + 0.0009*1j,
   0.2489 + 0.0518*1j, -0.0352 + 0.0213*1j, -0.0093 + 0.0020*1j, -0.0008 - 0.0019*1j, -0.0090 + 0.0114*1j,  0.0811 + 0.0212*1j,  0.0965 + 0.0000*1j, -0.0220 + 0.0001*1j,
  -0.0517 - 0.0112*1j,  0.0165 - 0.0033*1j,  0.0008 - 0.0003*1j, -0.0014 + 0.0003*1j,  0.0003 - 0.0071*1j, -0.0176 - 0.0009*1j, -0.0220 - 0.0001*1j,  0.0077 + 0.0000*1j, ]


w4 =  [0.0864 + 0.0000*1j, -0.0056 - 0.0156*1j, -0.0163 - 0.0154*1j,  0.0089 + 0.0006*1j,  0.0039 + 0.0069*1j,  0.1223 - 0.0028*1j,  0.1754 - 0.0500*1j, -0.0223 + 0.0017*1j,
  -0.0056 + 0.0156*1j,  0.0070 + 0.0000*1j, -0.0006 - 0.0100*1j, -0.0031 + 0.0002*1j, -0.0026 - 0.0108*1j, -0.0060 + 0.0251*1j, -0.0065 + 0.0306*1j,  0.0032 - 0.0178*1j,
  -0.0163 + 0.0154*1j, -0.0006 + 0.0100*1j,  0.0597 + 0.0000*1j,  0.0128 + 0.0036*1j,  0.0377 + 0.0021*1j, -0.0295 + 0.0071*1j, -0.0355 + 0.0425*1j,  0.0216 + 0.0168*1j,
   0.0089 - 0.0006*1j, -0.0031 - 0.0002*1j,  0.0128 - 0.0036*1j,  0.0129 + 0.0000*1j,  0.0082 + 0.0123*1j,  0.0017 + 0.0006*1j,  0.0042 + 0.0225*1j, -0.0027 + 0.0195*1j,
   0.0039 - 0.0069*1j, -0.0026 + 0.0108*1j,  0.0377 - 0.0021*1j,  0.0082 - 0.0123*1j,  0.0570 + 0.0000*1j,  0.0117 - 0.0032*1j,  0.0368 - 0.0121*1j,  0.0457 + 0.0129*1j,
   0.1223 + 0.0028*1j, -0.0060 - 0.0251*1j, -0.0295 - 0.0071*1j,  0.0017 - 0.0006*1j,  0.0117 + 0.0032*1j,  0.1984 + 0.0000*1j,  0.2787 - 0.0839*1j, -0.0311 - 0.0151*1j,
   0.1754 + 0.0500*1j, -0.0065 - 0.0306*1j, -0.0355 - 0.0425*1j,  0.0042 - 0.0225*1j,  0.0368 + 0.0121*1j,  0.2787 + 0.0839*1j,  0.5099 + 0.0000*1j,  0.0039 + 0.0005*1j,
  -0.0223 - 0.0017*1j,  0.0032 + 0.0178*1j,  0.0216 - 0.0168*1j, -0.0027 - 0.0195*1j,  0.0457 - 0.0129*1j, -0.0311 + 0.0151*1j,  0.0039 - 0.0005*1j,  0.0687 + 0.0000*1j, ]


w5 = [  0.1848 + 0.0000*1j,  0.0044 - 0.0004*1j,  0.0136 - 0.0075*1j, -0.0024 + 0.0082*1j,  0.0352 - 0.0297*1j,  0.2549 + 0.0077*1j,  0.2220 - 0.0569*1j, -0.0362 + 0.0016*1j,
   0.0044 + 0.0004*1j,  0.0132 + 0.0000*1j,  0.0088 + 0.0029*1j,  0.0046 + 0.0001*1j,  0.0181 - 0.0166*1j, -0.0034 + 0.0158*1j, -0.0026 + 0.0112*1j,  0.0083 + 0.0029*1j,
   0.0136 + 0.0075*1j,  0.0088 - 0.0029*1j,  0.0079 + 0.0000*1j,  0.0037 - 0.0008*1j,  0.0108 - 0.0133*1j,  0.0167 + 0.0238*1j,  0.0183 + 0.0157*1j,  0.0047 + 0.0015*1j,
  -0.0024 - 0.0082*1j,  0.0046 - 0.0001*1j,  0.0037 + 0.0008*1j,  0.0051 + 0.0000*1j,  0.0026 - 0.0026*1j, -0.0047 - 0.0031*1j, -0.0037 - 0.0007*1j,  0.0046 + 0.0107*1j,
   0.0352 + 0.0297*1j,  0.0181 + 0.0166*1j,  0.0108 + 0.0133*1j,  0.0026 + 0.0026*1j,  0.0608 + 0.0000*1j,  0.0195 + 0.0466*1j,  0.0289 + 0.0221*1j,  0.0121 + 0.0035*1j,
   0.2549 - 0.0077*1j, -0.0034 - 0.0158*1j,  0.0167 - 0.0238*1j, -0.0047 + 0.0031*1j,  0.0195 - 0.0466*1j,  0.3798 + 0.0000*1j,  0.3300 - 0.0894*1j, -0.0448 - 0.0053*1j,
   0.2220 + 0.0569*1j, -0.0026 - 0.0112*1j,  0.0183 - 0.0157*1j, -0.0037 + 0.0007*1j,  0.0289 - 0.0221*1j,  0.3300 + 0.0894*1j,  0.3127 + 0.0000*1j, -0.0349 - 0.0084*1j,
  -0.0362 - 0.0016*1j,  0.0083 - 0.0029*1j,  0.0047 - 0.0015*1j,  0.0046 - 0.0107*1j,  0.0121 - 0.0035*1j, -0.0448 + 0.0053*1j, -0.0349 + 0.0084*1j,  0.0358 + 0.0000*1j, ] 

w6 =   [ 0.3031 + 0.0000*1j, -0.0047 + 0.0211*1j, -0.0279 + 0.0034*1j, -0.0140 + 0.0061*1j,  0.0114 - 0.0253*1j,  0.2187 + 0.0270*1j,  0.3226 - 0.0629*1j,  0.0401 - 0.0211*1j,
  -0.0047 - 0.0211*1j,  0.0101 + 0.0000*1j,  0.0097 + 0.0037*1j, -0.0038 + 0.0060*1j,  0.0135 - 0.0139*1j, -0.0052 - 0.0202*1j, -0.0243 - 0.0202*1j,  0.0016 + 0.0027*1j,
  -0.0279 - 0.0034*1j,  0.0097 - 0.0037*1j,  0.0134 + 0.0000*1j, -0.0024 + 0.0063*1j,  0.0117 - 0.0196*1j, -0.0247 - 0.0125*1j, -0.0452 + 0.0047*1j,  0.0019 + 0.0095*1j,
  -0.0140 - 0.0061*1j, -0.0038 - 0.0060*1j, -0.0024 - 0.0063*1j,  0.0063 + 0.0000*1j, -0.0189 - 0.0005*1j, -0.0119 - 0.0005*1j, -0.0086 + 0.0047*1j,  0.0002 - 0.0056*1j,
   0.0114 + 0.0253*1j,  0.0135 + 0.0139*1j,  0.0117 + 0.0196*1j, -0.0189 + 0.0005*1j,  0.0732 + 0.0000*1j,  0.0243 + 0.0067*1j, -0.0016 + 0.0079*1j, -0.0159 + 0.0207*1j,
   0.2187 - 0.0270*1j, -0.0052 + 0.0202*1j, -0.0247 + 0.0125*1j, -0.0119 + 0.0005*1j,  0.0243 - 0.0067*1j,  0.1791 + 0.0000*1j,  0.2416 - 0.0793*1j,  0.0083 - 0.0153*1j,
   0.3226 + 0.0629*1j, -0.0243 + 0.0202*1j, -0.0452 - 0.0047*1j, -0.0086 - 0.0047*1j, -0.0016 - 0.0079*1j,  0.2416 + 0.0793*1j,  0.3885 + 0.0000*1j,  0.0339 - 0.0182*1j,
   0.0401 + 0.0211*1j,  0.0016 - 0.0027*1j,  0.0019 - 0.0095*1j,  0.0002 + 0.0056*1j, -0.0159 - 0.0207*1j,  0.0083 + 0.0153*1j,  0.0339 + 0.0182*1j,  0.0263 + 0.0000*1j, ]

BS1_output = [0.5080 + 0.0000*1j,   -0.0080 + 0.0665*1j,   -0.0273 + 0.0047*1j,    0.4364 + 0.0867*1j,    0.0062 + 0.0218*1j,    0.0155 - 0.0241*1j,    0.0136 + 0.0163*1j,    0.0099 - 0.0066*1j, 
  -0.0080 - 0.0665*1j,    0.0254 + 0.0000*1j,    0.0134 + 0.0189*1j,    0.0305 - 0.0541*1j,    0.0045 - 0.0037*1j,   -0.0016 - 0.0010*1j,    0.0006 - 0.0013*1j,   -0.0016 + 0.0006*1j, 
  -0.0273 - 0.0047*1j,    0.0134 - 0.0189*1j,    0.0285 + 0.0000*1j,    0.0029 - 0.0303*1j,   -0.0012 - 0.0053*1j,    0.0009 - 0.0001*1j,   -0.0009 + 0.0009*1j,    0.0005 + 0.0022*1j, 
   0.4364 - 0.0867*1j,    0.0305 + 0.0541*1j,    0.0029 + 0.0303*1j,    0.4331 + 0.0000*1j,    0.0113 + 0.0129*1j,    0.0122 - 0.0228*1j,    0.0125 + 0.0132*1j,    0.0068 - 0.0044*1j, 
   0.0062 - 0.0218*1j,    0.0045 + 0.0037*1j,   -0.0012 + 0.0053*1j,    0.0113 - 0.0129*1j,    0.0017 + 0.0000*1j,   -0.0007 - 0.0006*1j,    0.0006 - 0.0005*1j,   -0.0005 - 0.0004*1j, 
   0.0155 + 0.0241*1j,   -0.0016 + 0.0010*1j,    0.0009 + 0.0001*1j,    0.0122 + 0.0228*1j,   -0.0007 + 0.0006*1j,    0.0018 + 0.0000*1j,   -0.0005 + 0.0013*1j,    0.0006 + 0.0005*1j, 
   0.0136 - 0.0163*1j,    0.0006 + 0.0013*1j,   -0.0009 - 0.0009*1j,    0.0125 - 0.0132*1j,    0.0006 + 0.0005*1j,   -0.0005 - 0.0013*1j,    0.0010 + 0.0000*1j,    0.0002 - 0.0006*1j, 
   0.0099 + 0.0066*1j,   -0.0016 - 0.0006*1j,    0.0005 - 0.0022*1j,    0.0068 + 0.0044*1j,   -0.0005 + 0.0004*1j,    0.0006 - 0.0005*1j,    0.0002 + 0.0006*1j,    0.0005 + 0.0000*1j] 

BS2_output = [0.4674 + 0.0000*1j,   -0.0404 + 0.0074*1j,   -0.0302 - 0.0221*1j,   -0.0304 + 0.0097*1j,    0.0114 + 0.0382*1j,    0.4181 - 0.0292*1j,   -0.0146 + 0.0294*1j,    0.0296 - 0.0195*1j, 
  -0.0404 - 0.0074*1j,    0.0259 + 0.0000*1j,   -0.0067 + 0.0085*1j,    0.0142 + 0.0004*1j,   -0.0155 - 0.0093*1j,   -0.0481 - 0.0009*1j,    0.0082 - 0.0006*1j,   -0.0094 - 0.0013*1j, 
  -0.0302 + 0.0221*1j,   -0.0067 - 0.0085*1j,    0.0084 + 0.0000*1j,   -0.0031 - 0.0048*1j,    0.0016 + 0.0047*1j,   -0.0226 + 0.0226*1j,   -0.0026 - 0.0049*1j,    0.0009 + 0.0052*1j, 
  -0.0304 - 0.0097*1j,    0.0142 - 0.0004*1j,   -0.0031 + 0.0048*1j,    0.0136 + 0.0000*1j,   -0.0054 - 0.0029*1j,   -0.0332 + 0.0157*1j,    0.0055 - 0.0011*1j,   -0.0077 + 0.0006*1j, 
   0.0114 - 0.0382*1j,   -0.0155 + 0.0093*1j,    0.0016 - 0.0047*1j,   -0.0054 + 0.0029*1j,    0.0172 + 0.0000*1j,    0.0233 - 0.0311*1j,   -0.0026 + 0.0021*1j,    0.0038 - 0.0017*1j, 
   0.4181 + 0.0292*1j,   -0.0481 + 0.0009*1j,   -0.0226 - 0.0226*1j,   -0.0332 - 0.0157*1j,    0.0233 + 0.0311*1j,    0.4574 + 0.0000*1j,   -0.0186 + 0.0215*1j,    0.0343 - 0.0062*1j, 
  -0.0146 - 0.0294*1j,    0.0082 + 0.0006*1j,   -0.0026 + 0.0049*1j,    0.0055 + 0.0011*1j,   -0.0026 - 0.0021*1j,   -0.0186 - 0.0215*1j,    0.0044 + 0.0000*1j,   -0.0045 - 0.0014*1j, 
   0.0296 + 0.0195*1j,   -0.0094 + 0.0013*1j,    0.0009 - 0.0052*1j,   -0.0077 - 0.0006*1j,    0.0038 + 0.0017*1j,    0.0343 + 0.0062*1j,   -0.0045 + 0.0014*1j,    0.0058 + 0.0000*1] 

bs3_output = [0.0438 + 0.0000*1j,    0.0044 - 0.0042*1j,    0.0138 + 0.0121*1j,   -0.0042 - 0.0123*1j,    0.0360 + 0.0338*1j,    0.0042 - 0.0015*1j,    0.0119 - 0.0351*1j,    0.0030 - 0.0006*1j, 
   0.0044 + 0.0042*1j,    0.0048 + 0.0000*1j,    0.0179 + 0.0340*1j,    0.0009 - 0.0050*1j,    0.0218 + 0.0345*1j,    0.0015 + 0.0008*1j,    0.0008 - 0.0010*1j,    0.0037 + 0.0001*1j, 
   0.0138 - 0.0121*1j,    0.0179 - 0.0340*1j,    0.4374 + 0.0000*1j,   -0.0584 - 0.0115*1j,    0.4287 - 0.0617*1j,   -0.0094 - 0.0054*1j,   -0.0132 + 0.0351*1j,    0.0050 - 0.0400*1j, 
  -0.0042 + 0.0123*1j,    0.0009 + 0.0050*1j,   -0.0584 + 0.0115*1j,    0.0149 + 0.0000*1j,   -0.0619 + 0.0235*1j,    0.0048 + 0.0031*1j,    0.0086 - 0.0001*1j,    0.0019 + 0.0078*1j, 
   0.0360 - 0.0338*1j,    0.0218 - 0.0345*1j,    0.4287 + 0.0617*1j,   -0.0619 - 0.0235*1j,    0.4523 + 0.0000*1j,   -0.0061 - 0.0097*1j,   -0.0315 + 0.0077*1j,    0.0122 - 0.0391*1j, 
   0.0042 + 0.0015*1j,    0.0015 - 0.0008*1j,   -0.0094 + 0.0054*1j,    0.0048 - 0.0031*1j,   -0.0061 + 0.0097*1j,    0.0041 + 0.0000*1j,    0.0019 - 0.0046*1j,    0.0026 + 0.0019*1j, 
   0.0119 + 0.0351*1j,    0.0008 + 0.0010*1j,   -0.0132 - 0.0351*1j,    0.0086 + 0.0001*1j,   -0.0315 - 0.0077*1j,    0.0019 + 0.0046*1j,    0.0373 + 0.0000*1j,   -0.0035 + 0.0026*1j, 
   0.0030 + 0.0006*1j,    0.0037 - 0.0001*1j,    0.0050 + 0.0400*1j,    0.0019 - 0.0078*1j,    0.0122 + 0.0391*1j,    0.0026 - 0.0019*1j,   -0.0035 - 0.0026*1j,    0.0054 + 0.0000*1j]  

pps_output = [0.9976 + 0.0000*1j,   -0.0114 - 0.0055*1j,   -0.0013 - 0.0226*1j,   -0.0218 - 0.0048*1j,    0.0170 + 0.0187*1j,    0.0057 - 0.0007*1j,    0.0134 + 0.0189*1j,    0.0017 - 0.0038*1j, 
  -0.0114 + 0.0055*1j,    0.0002 + 0.0000*1j,    0.0001 + 0.0003*1j,    0.0003 - 0.0001*1j,   -0.0003 - 0.0001*1j,   -0.0001 + 0.0000*1j,   -0.0003 - 0.0001*1j,    0.0000 + 0.0001*1j, 
  -0.0013 + 0.0226*1j,    0.0001 - 0.0003*1j,    0.0005 + 0.0000*1j,    0.0001 - 0.0005*1j,   -0.0004 + 0.0004*1j,    0.0000 + 0.0001*1j,   -0.0004 + 0.0003*1j,    0.0001 + 0.0000*1j, 
  -0.0218 + 0.0048*1j,    0.0003 + 0.0001*1j,    0.0001 + 0.0005*1j,    0.0005 + 0.0000*1j,   -0.0005 - 0.0003*1j,   -0.0001 + 0.0000*1j,   -0.0004 - 0.0003*1j,   -0.0000 + 0.0001*1j, 
   0.0170 - 0.0187*1j,   -0.0003 + 0.0001*1j,   -0.0004 - 0.0004*1j,   -0.0005 + 0.0003*1j,    0.0006 + 0.0000*1j,    0.0001 - 0.0001*1j,    0.0006 + 0.0001*1j,   -0.0000 - 0.0001*1j, 
   0.0057 + 0.0007*1j,   -0.0001 - 0.0000*1j,    0.0000 - 0.0001*1j,   -0.0001 - 0.0000*1j,    0.0001 + 0.0001*1j,    0.0000 + 0.0000*1j,    0.0001 + 0.0001*1j,    0.0000 - 0.0000*1j, 
   0.0134 - 0.0189*1j,   -0.0003 + 0.0001*1j,   -0.0004 - 0.0003*1j,   -0.0004 + 0.0003*1j,    0.0006 - 0.0001*1j,    0.0001 - 0.0001*1j,    0.0005 + 0.0000*1j,   -0.0001 - 0.0001*1j, 
   0.0017 + 0.0038*1j,    0.0000 - 0.0001*1j,    0.0001 - 0.0000*1j,   -0.0000 - 0.0001*1j,   -0.0000 + 0.0001*1j,    0.0000 + 0.0000*1j,   -0.0001 + 0.0001*1j,    0.0000 + 0.0000*1j] 
 
Random_bs1_1_output = [0.0063 + 0.0000*1j,    0.0024 - 0.0030*1j,    0.0018 - 0.0003*1j,    0.0001 - 0.0012*1j,   -0.0108 - 0.0145*1j,   -0.0139 - 0.0102*1j,   -0.0085 - 0.0035*1j,   -0.0053 + 0.0019*1j, 
   0.0024 + 0.0030*1j,    0.0078 + 0.0000*1j,    0.0012 + 0.0013*1j,    0.0005 + 0.0023*1j,   -0.0016 - 0.0091*1j,   -0.0654 - 0.0301*1j,   -0.0125 - 0.0076*1j,   -0.0045 + 0.0002*1j, 
   0.0018 + 0.0003*1j,    0.0012 - 0.0013*1j,    0.0006 + 0.0000*1j,    0.0003 - 0.0001*1j,   -0.0026 - 0.0040*1j,   -0.0100 + 0.0015*1j,   -0.0032 - 0.0004*1j,   -0.0015 + 0.0006*1j, 
   0.0001 + 0.0012*1j,    0.0005 - 0.0023*1j,    0.0003 + 0.0001*1j,    0.0016 + 0.0000*1j,    0.0035 - 0.0001*1j,   -0.0065 + 0.0299*1j,   -0.0003 + 0.0038*1j,    0.0006 - 0.0002*1j, 
  -0.0108 + 0.0145*1j,   -0.0016 + 0.0091*1j,   -0.0026 + 0.0040*1j,    0.0035 + 0.0001*1j,    0.0558 + 0.0000*1j,    0.0951 + 0.0180*1j,    0.0308 - 0.0089*1j,    0.0065 - 0.0165*1j, 
  -0.0139 + 0.0102*1j,   -0.0654 + 0.0301*1j,   -0.0100 - 0.0015*1j,   -0.0065 - 0.0299*1j,    0.0951 - 0.0180*1j,    0.8856 + 0.0000*1j,    0.1622 - 0.0180*1j,    0.0206 - 0.0415*1j, 
  -0.0085 + 0.0035*1j,   -0.0125 + 0.0076*1j,   -0.0032 + 0.0004*1j,   -0.0003 - 0.0038*1j,    0.0308 + 0.0089*1j,    0.1622 + 0.0180*1j,    0.0361 + 0.0000*1j,    0.0084 - 0.0100*1j, 
  -0.0053 - 0.0019*1j,   -0.0045 - 0.0002*1j,   -0.0015 - 0.0006*1j,    0.0006 + 0.0002*1j,    0.0065 + 0.0165*1j,    0.0206 + 0.0415*1j,    0.0084 + 0.0100*1j,    0.0061 + 0.0000*1j] 

random_bs1_2_output = [0.0029 + 0.0000*1j,    0.0007 + 0.0005*1j,    0.0002 + 0.0002*1j,    0.0002 + 0.0004*1j,    0.0035 - 0.0079*1j,    0.0023 - 0.0000*1j,    0.0024 - 0.0022*1j,    0.0009 + 0.0008*1j, 
   0.0007 - 0.0005*1j,    0.0048 + 0.0000*1j,    0.0007 + 0.0004*1j,   -0.0023 - 0.0018*1j,   -0.0049 - 0.0062*1j,    0.0511 - 0.0415*1j,    0.0080 - 0.0089*1j,    0.0016 + 0.0001*1j, 
   0.0002 - 0.0002*1j,    0.0007 - 0.0004*1j,    0.0002 + 0.0000*1j,   -0.0005 - 0.0000*1j,   -0.0013 - 0.0010*1j,    0.0037 - 0.0105*1j,    0.0004 - 0.0022*1j,    0.0003 - 0.0001*1j, 
   0.0002 - 0.0004*1j,   -0.0023 + 0.0018*1j,   -0.0005 + 0.0000*1j,    0.0021 + 0.0000*1j,    0.0029 - 0.0008*1j,   -0.0102 + 0.0414*1j,   -0.0011 + 0.0069*1j,   -0.0005 + 0.0004*1j, 
   0.0035 + 0.0079*1j,   -0.0049 + 0.0062*1j,   -0.0013 + 0.0010*1j,    0.0029 + 0.0008*1j,    0.0333 + 0.0000*1j,   -0.0124 + 0.0861*1j,    0.0078 + 0.0178*1j,   -0.0022 + 0.0045*1j, 
   0.0023 + 0.0000*1j,    0.0511 + 0.0415*1j,    0.0037 + 0.0105*1j,   -0.0102 - 0.0414*1j,   -0.0124 - 0.0861*1j,    0.9250 + 0.0000*1j,    0.1599 - 0.0199*1j,    0.0136 + 0.0119*1j, 
   0.0024 + 0.0022*1j,    0.0080 + 0.0089*1j,    0.0004 + 0.0022*1j,   -0.0011 - 0.0069*1j,    0.0078 - 0.0178*1j,    0.1599 + 0.0199*1j,    0.0310 + 0.0000*1j,    0.0022 + 0.0035*1j, 
   0.0009 - 0.0008*1j,    0.0016 - 0.0001*1j,    0.0003 + 0.0001*1j,   -0.0005 - 0.0004*1j,   -0.0022 - 0.0045*1j,    0.0136 - 0.0119*1j,    0.0022 - 0.0035*1j,    0.0008 + 0.0000*1j] 
   
random_bs1_4_output =  [0.0066 + 0.0000*1j,    0.0031 + 0.0022*1j,    0.0039 - 0.0051*1j,   -0.0022 - 0.0003*1j,   -0.0140 - 0.0307*1j,    0.0004 - 0.0257*1j,   -0.0325 - 0.0477*1j,   -0.0069 - 0.0055*1j, 
   0.0031 - 0.0022*1j,    0.0023 + 0.0000*1j,   -0.0003 - 0.0031*1j,   -0.0013 + 0.0004*1j,   -0.0158 - 0.0107*1j,   -0.0070 - 0.0125*1j,   -0.0261 - 0.0112*1j,   -0.0045 - 0.0003*1j, 
   0.0039 + 0.0051*1j,   -0.0003 + 0.0031*1j,    0.0110 + 0.0000*1j,   -0.0010 - 0.0003*1j,    0.0083 - 0.0291*1j,    0.0141 - 0.0195*1j,    0.0030 - 0.0775*1j,   -0.0019 - 0.0114*1j, 
  -0.0022 + 0.0003*1j,   -0.0013 - 0.0004*1j,   -0.0010 + 0.0003*1j,    0.0012 + 0.0000*1j,    0.0058 + 0.0116*1j,   -0.0005 + 0.0103*1j,    0.0048 + 0.0186*1j,    0.0016 + 0.0021*1j, 
  -0.0140 + 0.0307*1j,   -0.0158 + 0.0107*1j,    0.0083 + 0.0291*1j,    0.0058 - 0.0116*1j,    0.1822 + 0.0000*1j,    0.1274 + 0.0627*1j,    0.3119 - 0.0134*1j,    0.0434 - 0.0163*1j, 
   0.0004 + 0.0257*1j,   -0.0070 + 0.0125*1j,    0.0141 + 0.0195*1j,   -0.0005 - 0.0103*1j,    0.1274 - 0.0627*1j,    0.1118 + 0.0000*1j,    0.2251 - 0.1120*1j,    0.0264 - 0.0257*1j, 
  -0.0325 + 0.0477*1j,   -0.0261 + 0.0112*1j,    0.0030 + 0.0775*1j,    0.0048 - 0.0186*1j,    0.3119 + 0.0134*1j,    0.2251 + 0.1120*1j,    0.6704 + 0.0000*1j,    0.0941 - 0.0249*1j, 
  -0.0069 + 0.0055*1j,   -0.0045 + 0.0003*1j,   -0.0019 + 0.0114*1j,    0.0016 - 0.0021*1j,    0.0434 + 0.0163*1j,    0.0264 + 0.0257*1j,    0.0941 + 0.0249*1j,    0.0144 + 0.0000*1j] 
 
random_bs1_5_output = [0.0345 + 0.0000*1j,    0.0067 + 0.0105*1j,    0.0095 - 0.0038*1j,   -0.0003 + 0.0029*1j,    0.0765 + 0.0537*1j,    0.0127 + 0.0231*1j,    0.0222 + 0.0102*1j,   -0.0004 - 0.0001*1j, 
   0.0067 - 0.0105*1j,    0.0059 + 0.0000*1j,    0.0028 - 0.0031*1j,    0.0016 + 0.0004*1j,    0.0218 + 0.0138*1j,    0.0051 + 0.0083*1j,    0.0054 + 0.0044*1j,    0.0001 + 0.0008*1j, 
   0.0095 + 0.0038*1j,    0.0028 + 0.0031*1j,    0.0065 + 0.0000*1j,    0.0007 + 0.0000*1j,    0.0114 + 0.0663*1j,   -0.0026 + 0.0209*1j,    0.0056 + 0.0196*1j,    0.0005 + 0.0008*1j, 
  -0.0003 - 0.0029*1j,    0.0016 - 0.0004*1j,    0.0007 - 0.0000*1j,    0.0008 + 0.0000*1j,   -0.0066 + 0.0067*1j,   -0.0021 + 0.0023*1j,   -0.0022 + 0.0030*1j,    0.0000 + 0.0005*1j, 
   0.0765 - 0.0537*1j,    0.0218 - 0.0138*1j,    0.0114 - 0.0663*1j,   -0.0066 - 0.0067*1j,    0.8001 + 0.0000*1j,    0.2330 + 0.0617*1j,    0.2456 - 0.0356*1j,    0.0098 - 0.0082*1j, 
   0.0127 - 0.0231*1j,    0.0051 - 0.0083*1j,   -0.0026 - 0.0209*1j,   -0.0021 - 0.0023*1j,    0.2330 - 0.0617*1j,    0.0740 + 0.0000*1j,    0.0694 - 0.0284*1j,    0.0027 - 0.0030*1j, 
   0.0222 - 0.0102*1j,    0.0054 - 0.0044*1j,    0.0056 - 0.0196*1j,   -0.0022 - 0.0030*1j,    0.2456 + 0.0356*1j,    0.0694 + 0.0284*1j,    0.0779 + 0.0000*1j,    0.0037 - 0.0023*1j, 
  -0.0004 + 0.0001*1j,    0.0001 - 0.0008*1j,    0.0005 - 0.0008*1j,    0.0000 - 0.0005*1j,    0.0098 + 0.0082*1j,    0.0027 + 0.0030*1j,    0.0037 + 0.0023*1j,    0.0004 + 0.0000*1j] 

random_bs2_1_output = [0.4032 + 0.0000*1j,    0.0449 - 0.0467*1j,   -0.0197 - 0.0055*1j,    0.0239 - 0.0162*1j,    0.0100 - 0.0515*1j,    0.4384 + 0.0321*1j,   -0.0083 - 0.0313*1j,    0.0006 - 0.0035*1j, 
   0.0449 + 0.0467*1j,    0.0268 + 0.0000*1j,    0.0009 - 0.0025*1j,    0.0042 + 0.0049*1j,    0.0152 - 0.0270*1j,    0.0340 + 0.0738*1j,    0.0018 - 0.0045*1j,   -0.0022 - 0.0036*1j, 
  -0.0197 + 0.0055*1j,    0.0009 + 0.0025*1j,    0.0024 + 0.0000*1j,    0.0002 + 0.0012*1j,    0.0009 + 0.0002*1j,   -0.0223 + 0.0081*1j,    0.0016 + 0.0022*1j,    0.0012 + 0.0012*1j, 
   0.0239 + 0.0162*1j,    0.0042 - 0.0049*1j,    0.0002 - 0.0012*1j,    0.0044 + 0.0000*1j,   -0.0034 - 0.0029*1j,    0.0302 + 0.0227*1j,    0.0013 - 0.0007*1j,    0.0005 + 0.0029*1j, 
   0.0100 + 0.0515*1j,    0.0152 + 0.0270*1j,    0.0009 - 0.0002*1j,   -0.0034 + 0.0029*1j,    0.0426 + 0.0000*1j,   -0.0247 + 0.0504*1j,    0.0043 - 0.0039*1j,    0.0052 - 0.0069*1j, 
   0.4384 - 0.0321*1j,    0.0340 - 0.0738*1j,   -0.0223 - 0.0081*1j,    0.0302 - 0.0227*1j,   -0.0247 - 0.0504*1j,    0.5107 + 0.0000*1j,   -0.0099 - 0.0321*1j,    0.0003 + 0.0017*1j, 
  -0.0083 + 0.0313*1j,    0.0018 + 0.0045*1j,    0.0016 - 0.0022*1j,    0.0013 + 0.0007*1j,    0.0043 + 0.0039*1j,   -0.0099 + 0.0321*1j,    0.0041 + 0.0000*1j,    0.0030 + 0.0002*1j, 
   0.0006 + 0.0035*1j,   -0.0022 + 0.0036*1j,    0.0012 - 0.0012*1j,    0.0005 - 0.0029*1j,    0.0052 + 0.0069*1j,    0.0003 - 0.0017*1j,    0.0030 - 0.0002*1j,    0.0058 + 0.0000*1j] 

random_bs2_2_output = [0.4026 + 0.0000*1j,    0.0714 + 0.0542*1j,   -0.0325 - 0.0108*1j,   -0.0286 - 0.0064*1j,   -0.0020 - 0.0184*1j,    0.4102 + 0.0636*1j,    0.0095 - 0.0262*1j,    0.0204 - 0.0361*1j, 
   0.0714 - 0.0542*1j,    0.0396 + 0.0000*1j,   -0.0071 + 0.0041*1j,   -0.0029 + 0.0025*1j,   -0.0240 - 0.0192*1j,    0.0905 - 0.0128*1j,   -0.0016 - 0.0068*1j,    0.0046 - 0.0091*1j, 
  -0.0325 + 0.0108*1j,   -0.0071 - 0.0041*1j,    0.0059 + 0.0000*1j,    0.0018 + 0.0023*1j,   -0.0018 + 0.0034*1j,   -0.0286 + 0.0066*1j,    0.0016 - 0.0001*1j,    0.0014 + 0.0084*1j, 
  -0.0286 + 0.0064*1j,   -0.0029 - 0.0025*1j,    0.0018 - 0.0023*1j,    0.0055 + 0.0000*1j,   -0.0022 - 0.0006*1j,   -0.0287 + 0.0032*1j,   -0.0030 + 0.0007*1j,    0.0049 - 0.0002*1j, 
  -0.0020 + 0.0184*1j,   -0.0240 + 0.0192*1j,   -0.0018 - 0.0034*1j,   -0.0022 + 0.0006*1j,    0.0372 + 0.0000*1j,   -0.0414 - 0.0082*1j,    0.0009 + 0.0024*1j,   -0.0049 + 0.0039*1j, 
   0.4102 - 0.0636*1j,    0.0905 + 0.0128*1j,   -0.0286 - 0.0066*1j,   -0.0287 - 0.0032*1j,   -0.0414 + 0.0082*1j,    0.4864 + 0.0000*1j,    0.0053 - 0.0328*1j,    0.0227 - 0.0434*1j, 
   0.0095 + 0.0262*1j,   -0.0016 + 0.0068*1j,    0.0016 + 0.0001*1j,   -0.0030 - 0.0007*1j,    0.0009 - 0.0024*1j,    0.0053 + 0.0328*1j,    0.0051 + 0.0000*1j,   -0.0006 + 0.0058*1j, 
   0.0204 + 0.0361*1j,    0.0046 + 0.0091*1j,    0.0014 - 0.0084*1j,    0.0049 + 0.0002*1j,   -0.0049 - 0.0039*1j,    0.0227 + 0.0434*1j,   -0.0006 - 0.0058*1j,    0.0178 + 0.0000*1j] 
   
random_bs2_3_output = [0.3910 + 0.0000*1j,    0.0699 + 0.0552*1j,   -0.0288 - 0.0304*1j,   -0.0100 - 0.0298*1j,   -0.0009 - 0.0194*1j,    0.4079 + 0.0460*1j,    0.0079 - 0.0083*1j,    0.0454 - 0.0303*1j, 
   0.0699 - 0.0552*1j,    0.0391 + 0.0000*1j,   -0.0077 - 0.0007*1j,   -0.0072 - 0.0038*1j,   -0.0194 - 0.0238*1j,    0.0885 - 0.0213*1j,   -0.0004 - 0.0065*1j,    0.0064 - 0.0040*1j, 
  -0.0288 + 0.0304*1j,   -0.0077 + 0.0007*1j,    0.0080 + 0.0000*1j,    0.0005 + 0.0035*1j,   -0.0013 + 0.0013*1j,   -0.0270 + 0.0319*1j,    0.0012 - 0.0031*1j,    0.0009 + 0.0135*1j, 
  -0.0100 + 0.0298*1j,   -0.0072 + 0.0038*1j,    0.0005 - 0.0035*1j,    0.0056 + 0.0000*1j,    0.0037 + 0.0012*1j,   -0.0170 + 0.0242*1j,   -0.0030 + 0.0032*1j,    0.0044 - 0.0023*1j, 
  -0.0009 + 0.0194*1j,   -0.0194 + 0.0238*1j,   -0.0013 - 0.0013*1j,    0.0037 - 0.0012*1j,    0.0382 + 0.0000*1j,   -0.0421 + 0.0034*1j,    0.0034 + 0.0034*1j,   -0.0069 - 0.0036*1j, 
   0.4079 - 0.0460*1j,    0.0885 + 0.0213*1j,   -0.0270 - 0.0319*1j,   -0.0170 - 0.0242*1j,   -0.0421 - 0.0034*1j,    0.4845 + 0.0000*1j,    0.0015 - 0.0168*1j,    0.0615 - 0.0274*1j, 
   0.0079 + 0.0083*1j,   -0.0004 + 0.0065*1j,    0.0012 + 0.0031*1j,   -0.0030 - 0.0032*1j,    0.0034 - 0.0034*1j,    0.0015 + 0.0168*1j,    0.0065 + 0.0000*1j,   -0.0082 + 0.0051*1j, 
   0.0454 + 0.0303*1j,    0.0064 + 0.0040*1j,    0.0009 - 0.0135*1j,    0.0044 + 0.0023*1j,   -0.0069 + 0.0036*1j,    0.0615 + 0.0274*1j,   -0.0082 - 0.0051*1j,    0.0269 + 0.0000*1j] 
 
random_bs2_4_output = [0.1036 + 0.0000*1j,   -0.0001 - 0.0032*1j,    0.0119 - 0.0191*1j,   -0.0092 + 0.0060*1j,    0.0176 - 0.0185*1j,    0.2480 - 0.0090*1j,    0.0013 - 0.0168*1j,    0.0191 - 0.0088*1j, 
  -0.0001 + 0.0032*1j,    0.0034 + 0.0000*1j,   -0.0044 + 0.0025*1j,    0.0072 - 0.0026*1j,    0.0026 - 0.0026*1j,    0.0027 + 0.0105*1j,    0.0036 - 0.0004*1j,   -0.0097 - 0.0042*1j, 
   0.0119 + 0.0191*1j,   -0.0044 - 0.0025*1j,    0.0237 + 0.0000*1j,   -0.0123 - 0.0020*1j,   -0.0009 + 0.0082*1j,    0.0265 + 0.0400*1j,   -0.0157 + 0.0042*1j,    0.0420 + 0.0183*1j, 
  -0.0092 - 0.0060*1j,    0.0072 + 0.0026*1j,   -0.0123 + 0.0020*1j,    0.0246 + 0.0000*1j,    0.0012 - 0.0037*1j,   -0.0183 + 0.0020*1j,    0.0002 + 0.0190*1j,   -0.0017 - 0.0206*1j, 
   0.0176 + 0.0185*1j,    0.0026 + 0.0026*1j,   -0.0009 - 0.0082*1j,    0.0012 + 0.0037*1j,    0.0135 + 0.0000*1j,    0.0435 + 0.0434*1j,    0.0128 - 0.0032*1j,   -0.0063 - 0.0202*1j, 
   0.2480 + 0.0090*1j,    0.0027 - 0.0105*1j,    0.0265 - 0.0400*1j,   -0.0183 - 0.0020*1j,    0.0435 - 0.0434*1j,    0.6104 + 0.0000*1j,    0.0307 - 0.0356*1j,    0.0292 - 0.0377*1j, 
   0.0013 + 0.0168*1j,    0.0036 + 0.0004*1j,   -0.0157 - 0.0042*1j,    0.0002 - 0.0190*1j,    0.0128 + 0.0032*1j,    0.0307 + 0.0356*1j,    0.0750 + 0.0000*1j,   -0.0575 - 0.0615*1j, 
   0.0191 + 0.0088*1j,   -0.0097 + 0.0042*1j,    0.0420 - 0.0183*1j,   -0.0017 + 0.0206*1j,   -0.0063 + 0.0202*1j,    0.0292 + 0.0377*1j,   -0.0575 + 0.0615*1j,    0.1459 + 0.0000*1j] 

random_bs2_5_output = [0.2140 + 0.0000*1j,   -0.0519 - 0.0266*1j,   -0.0358 + 0.0148*1j,    0.0080 + 0.0013*1j,    0.0082 + 0.0360*1j,    0.3443 + 0.0219*1j,    0.0137 - 0.0137*1j,    0.0051 - 0.0415*1j, 
  -0.0519 + 0.0266*1j,    0.0164 + 0.0000*1j,    0.0088 - 0.0080*1j,   -0.0013 + 0.0000*1j,   -0.0083 - 0.0055*1j,   -0.0884 + 0.0381*1j,   -0.0038 + 0.0094*1j,    0.0036 + 0.0145*1j, 
  -0.0358 - 0.0148*1j,    0.0088 + 0.0080*1j,    0.0157 + 0.0000*1j,    0.0023 - 0.0038*1j,    0.0022 + 0.0023*1j,   -0.0629 - 0.0217*1j,   -0.0142 + 0.0212*1j,   -0.0043 + 0.0227*1j, 
   0.0080 - 0.0013*1j,   -0.0013 - 0.0000*1j,    0.0023 + 0.0038*1j,    0.0029 + 0.0000*1j,   -0.0037 + 0.0069*1j,    0.0072 - 0.0014*1j,   -0.0109 + 0.0035*1j,   -0.0060 + 0.0051*1j, 
   0.0082 - 0.0360*1j,   -0.0083 + 0.0055*1j,    0.0022 - 0.0023*1j,   -0.0037 - 0.0069*1j,    0.0744 + 0.0000*1j,    0.0396 - 0.0316*1j,    0.0104 + 0.0126*1j,    0.0146 - 0.0048*1j, 
   0.3443 - 0.0219*1j,   -0.0884 - 0.0381*1j,   -0.0629 + 0.0217*1j,    0.0072 + 0.0014*1j,    0.0396 + 0.0316*1j,    0.5760 + 0.0000*1j,    0.0409 - 0.0294*1j,    0.0150 - 0.0826*1j, 
   0.0137 + 0.0137*1j,   -0.0038 - 0.0094*1j,   -0.0142 - 0.0212*1j,   -0.0109 - 0.0035*1j,    0.0104 - 0.0126*1j,    0.0409 + 0.0294*1j,    0.0617 + 0.0000*1j,    0.0397 - 0.0209*1j, 
   0.0051 + 0.0415*1j,    0.0036 - 0.0145*1j,   -0.0043 - 0.0227*1j,   -0.0060 - 0.0051*1j,    0.0146 + 0.0048*1j,    0.0150 + 0.0826*1j,    0.0397 + 0.0209*1j,    0.0388 + 0.0000*1j]
  
random_bs2_6_output = [0.6983 + 0.0000*1j,   -0.0088 + 0.0465*1j,    0.0273 + 0.0198*1j,   -0.0299 + 0.0289*1j,    0.0959 + 0.0571*1j,    0.3793 - 0.0056*1j,    0.0285 - 0.0433*1j,    0.0258 - 0.0299*1j, 
  -0.0088 - 0.0465*1j,    0.0175 + 0.0000*1j,    0.0004 - 0.0004*1j,    0.0071 + 0.0049*1j,   -0.0225 - 0.0082*1j,   -0.0051 - 0.0218*1j,   -0.0061 + 0.0029*1j,    0.0004 - 0.0009*1j, 
   0.0273 - 0.0198*1j,    0.0004 + 0.0004*1j,    0.0019 + 0.0000*1j,   -0.0001 + 0.0013*1j,    0.0062 + 0.0026*1j,    0.0151 - 0.0111*1j,    0.0005 - 0.0023*1j,    0.0001 - 0.0022*1j, 
  -0.0299 - 0.0289*1j,    0.0071 - 0.0049*1j,   -0.0001 - 0.0013*1j,    0.0049 + 0.0000*1j,   -0.0105 - 0.0011*1j,   -0.0157 - 0.0143*1j,   -0.0030 + 0.0028*1j,   -0.0013 - 0.0003*1j, 
   0.0959 - 0.0571*1j,   -0.0225 + 0.0082*1j,    0.0062 - 0.0026*1j,   -0.0105 + 0.0011*1j,    0.0622 + 0.0000*1j,    0.0514 - 0.0377*1j,    0.0051 - 0.0161*1j,   -0.0036 - 0.0068*1j, 
   0.3793 + 0.0056*1j,   -0.0051 + 0.0218*1j,    0.0151 + 0.0111*1j,   -0.0157 + 0.0143*1j,    0.0514 + 0.0377*1j,    0.2068 + 0.0000*1j,    0.0168 - 0.0226*1j,    0.0144 - 0.0167*1j, 
   0.0285 + 0.0433*1j,   -0.0061 - 0.0029*1j,    0.0005 + 0.0023*1j,   -0.0030 - 0.0028*1j,    0.0051 + 0.0161*1j,    0.0168 + 0.0226*1j,    0.0057 + 0.0000*1j,    0.0025 - 0.0005*1j, 
   0.0258 + 0.0299*1j,    0.0004 + 0.0009*1j,    0.0001 + 0.0022*1j,   -0.0013 + 0.0003*1j,   -0.0036 + 0.0068*1j,    0.0144 + 0.0167*1j,    0.0025 + 0.0005*1j,    0.0027 + 0.0000*1j] 


f1 = 0.8978
f2 = 0.8913
f3 = 0.9669
f4 = 0.9253
f5 = 0.9628
f6 = 0.9584  

'''
pps_state = [0.9976 + 0.0000*1j,  -0.0114 - 0.0055*1j,  -0.0013 - 0.0226*1j,  -0.0218 - 0.0048*1j,   0.0170 + 0.0187*1j,   0.0057 - 0.0007*1j,   0.0134 + 0.0189*1j,   0.0017 - 0.0038*1j,
  -0.0114 + 0.0055*1j,   0.0002 + 0.0000*1j,   0.0001 + 0.0003*1j,   0.0003 - 0.0001*1j,  -0.0003 - 0.0001*1j,  -0.0001 + 0.0000*1j,  -0.0003 - 0.0001*1j,   0.0000 + 0.0001*1j,
  -0.0013 + 0.0226*1j,   0.0001 - 0.0003*1j,   0.0005 + 0.0000*1j,   0.0001 - 0.0005*1j,  -0.0004 + 0.0004*1j,   0.0000 + 0.0001*1j,  -0.0004 + 0.0003*1j,   0.0001 + 0.0000*1j,
  -0.0218 + 0.0048*1j,   0.0003 + 0.0001*1j,   0.0001 + 0.0005*1j,   0.0005 + 0.0000*1j,  -0.0005 - 0.0003*1j,  -0.0001 + 0.0000*1j,  -0.0004 - 0.0003*1j,  -0.0000 + 0.0001*1j,
   0.0170 - 0.0187*1j,  -0.0003 + 0.0001*1j,  -0.0004 - 0.0004*1j,  -0.0005 + 0.0003*1j,   0.0006 + 0.0000*1j,   0.0001 - 0.0001*1j,   0.0006 + 0.0001*1j,  -0.0000 - 0.0001*1j,
   0.0057 + 0.0007*1j,  -0.0001 - 0.0000*1j,   0.0000 - 0.0001*1j,  -0.0001 - 0.0000*1j,   0.0001 + 0.0001*1j,   0.0000 + 0.0000*1j,   0.0001 + 0.0001*1j,   0.0000 - 0.0000*1j,
   0.0134 - 0.0189*1j,  -0.0003 + 0.0001*1j,  -0.0004 - 0.0003*1j,  -0.0004 + 0.0003*1j,   0.0006 - 0.0001*1j,   0.0001 - 0.0001*1j,   0.0005 + 0.0000*1j,  -0.0001 - 0.0001*1j,
   0.0017 + 0.0038*1j,   0.0000 - 0.0001*1j,   0.0001 - 0.0000*1j,  -0.0000 - 0.0001*1j,  -0.0000 + 0.0001*1j,   0.0000 + 0.0000*1j,  -0.0001 + 0.0001*1j,   0.0000 + 0.0000*1]

pps_fidelity = 0.9976  
'''

list_fid = [0.9612,0.9807,0.9598,0.9635,0.9544,0.9323,0.9493,0.9485,0.8743,0.7646,0.9549,0.9755,0.9585,0.9329,0.9482,0.9848,0.9447,0.9718,0.9753,0.9658,0.9976,0.9705,0.9807,0.9381,0.9731,0.9687,0.9538,0.9514,0.9262,0.9246,0.9489,f1,f2,f3,f4,f5,f6]
list_states = [random_bs3_1,random_bs3_2,random_bs3_3,random_bs3_4,random_bs3_5, random_ghz_1,random_ghz_2,random_ghz_3, random_ghz_4,random_ghz_5,random_ghz_6 ,random_sep_2 ,random_sep_3,random_sep_4,random_sep_5, sep_output,
              BS1_output,BS2_output,bs3_output,pps_output,Random_bs1_1_output,random_bs1_2_output,random_bs1_4_output,random_bs1_5_output,random_bs2_1_output,random_bs2_2_output, random_bs2_3_output,random_bs2_4_output,random_bs2_5_output,random_bs2_6_output, W_output, w1,w2,w3,w4,w5]


code_3 = list_states[0:5]
code_3.append(list_states[18])
code_3_fid = list_fid[0:5]
code_3_fid.append(list_fid[18])

code_5 = list_states[5:11]
code_5_fid = list_fid[5:11]

code_0 = list_states[11:16]
#code_0.append(pps_state)
code_0_fid = list_fid[11:16]
#code_0_fid.append(pps_fidelity)

code_1 = list_states[21:25]
code_1.append(list_states[16])
code_1_fid = list_fid[21:25]
code_1_fid.append(list_fid[16])

code_2 = list_states[25:30]
code_2.append(list_states[17])
code_2_fid = list_fid[25:30]
code_2_fid.append(list_fid[17])

code_4 = list_states[30:36]
code_4_fid = list_fid[30:36]

list_all_states = [code_0, code_1, code_2, code_3, code_4, code_5]
fid_all_states = [code_0_fid, code_1_fid, code_2_fid, code_3_fid, code_4_fid, code_5_fid]

code_0_list = []
code_1_list = []
code_2_list = []
code_3_list = []
code_4_list = []
code_5_list = []

code_all_list = [code_0_list, code_1_list, code_2_list,code_3_list,code_4_list, code_5_list]

for i in range(6):
    for j in range(len(list_all_states[i])):      
        #Matrix = (np.array(list_all_states[i][j])).reshape((8,8))
        Matrix = (np.array(list_all_states[i][j]))
        Matrix_list = array_to_label_list(Matrix= Matrix,n=3,k=i)
        Matrix_list.append(fid_all_states[i][j])
        code_all_list[i].append(Matrix_list)
        
code_all_df = []

cols = labels(3)
cols.append('FIDELITY')

for i in range(6):
    code_all_df.append(pd.DataFrame(code_all_list[i]))
    
all_df = pd.concat(code_all_df, axis = 0)
all_df.columns = cols

all_df = all_df.reset_index()
all_df = all_df.drop(['index'],axis=1)

bad_fid = [0.7646, 0.8913, 0.9612 , 0.9246 ]
for i in range(len(bad_fid)):
    all_df = all_df[all_df['FIDELITY'] != bad_fid[i]]
    
df_0 = all_df.loc[all_df['CODE'] == 0]
df_1 = all_df.loc[all_df['CODE'] == 1]
df_2 = all_df.loc[all_df['CODE'] == 2]
df_3 = all_df.loc[all_df['CODE'] == 3]
df_4 = all_df.loc[all_df['CODE'] == 4]
df_5 = all_df.loc[all_df['CODE'] == 5]

#df_list = [df_0, df_1 , df_2, df_3, df_4, df_5]

#df_train = pd.concat([df_list[i].iloc[:3:] for i in range(6)], axis = 0)
#df_test = pd.concat([df_list[i].iloc[3::] for i in range(6)], axis = 0)

In [4]:
#,sep
theo_code_0_1=[[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0.961538,0.192308,0,0],[0,0,0,0,0.192308,0.0384615,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0]]
theo_code_0_2=[[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0.64,0,0.48,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0.48,0,0.36,0],[0,0,0,0,0,0,0,0]]
theo_code_0_3=[[0.264706,0,0,0,0.44117,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.44117,0,0,0,0.73529,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0]]
theo_code_0_4=[[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0.692308,0.461538,0,0],[0,0,0,0,0.461538,0.307692,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0]]
theo_code_0_5=[[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0.1,0.3,0,0],[0,0,0,0,0.3,0.9,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0]]

#,bs1,
theo_code_1_1=[[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0.941176,0.235294,0],[0,0,0,0,0,0.235294,0.0588235,0],[0,0,0,0,0,0,0,0]]
theo_code_1_2=[[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0.952941,0.211765,0],[0,0,0,0,0,0.211765,0.0470588,0],[0,0,0,0,0,0,0,0]]
theo_code_1_3=[[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0.862069,0.344828,0],[0,0,0,0,0,0.344828,0.137931,0],[0,0,0,0,0,0,0,0]]
theo_code_1_4=[[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0.166666,0.166666,0.333333,0],[0,0,0,0,0.166666,0.166666,0.333333,0],[0,0,0,0,0.333333,0.333333,0.666666,0],[0,0,0,0,0,0,0,0]]
theo_code_1_5=[[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0.818181,0.272727,0.272727,0],[0,0,0,0,0.272727,0.090909,0.090909,0],[0,0,0,0,0.272727,0.090909,0.090909,0],[0,0,0,0,0,0,0,0]]

#,bs2,
theo_code_2_1=[[0.409836,0,0,0,0,0.491803,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.491803,0,0,0,0,0.590164,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0]]
theo_code_2_2=[[0.4235294,0,0,0,0,0.494117,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.494117,0,0,0,0,0.5764705,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0]]
theo_code_2_3=[[0.441379,0,0,0,0,0.496551,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.496551,0,0,0,0,0.5586206,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0]]
theo_code_2_4=[[0.2,0,0,0,0,0.4,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.4,0,0,0,0,0.8,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0]]
theo_code_2_5=[[0.36,0,0,0,0,0.48,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.48,0,0,0,0,0.64,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0]]

#,bs3,
theo_code_3_1=[[0.8,0,0,0,0,0,0.4,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.4,0,0,0,0,0,0.2,0],[0,0,0,0,0,0,0,0]]
theo_code_3_2=[[0.735294,0,0,0,0,0,0.4411764,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.4411764,0,0,0,0,0,0.264706,0],[0,0,0,0,0,0,0,0]]
theo_code_3_3=[[0.307692,0,0,0,0,0,0.461538,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.461538,0,0,0,0,0,0.692307,0],[0,0,0,0,0,0,0,0]]
theo_code_3_4=[[0.409836,0,0,0,0,0,0.491803,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.491803,0,0,0,0,0,0.590164,0],[0,0,0,0,0,0,0,0]]
theo_code_3_5=[[0.390244,0,0,0,0,0,0.4878048,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.4878048,0,0,0,0,0,0.609756,0],[0,0,0,0,0,0,0,0]]

#,w,
theo_code_4_1=[[0.111111,0,0,0,0,0.222222,0.222222,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.222222,0,0,0,0,0.444444,0.444444,0],[0.222222,0,0,0,0,0.444444,0.444444,0],[0,0,0,0,0,0,0,0]]
theo_code_4_2=[[0.653061,0,0,0,0,0.244898,0.408163,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.244898,0,0,0,0,0.091836,0.153061,0],[0.408163,0,0,0,0,0.153061,0.255102,0],[0,0,0,0,0,0,0,0]]
theo_code_4_3=[[0.757575,0,0,0,0,0.30303,0.30303,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.30303,0,0,0,0,0.121212,0.121212,0],[0.30303,0,0,0,0,0.121212,0.121212,0],[0,0,0,0,0,0,0,0]]
theo_code_4_4=[[0.16129,0,0,0,0,0.225806,0.290322,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.225806,0,0,0,0,0.316129,0.406451,0],[0.290322,0,0,0,0,0.406451,0.52258,0],[0,0,0,0,0,0,0,0]]
theo_code_4_5=[[0.4090909,0,0,0,0,0.272727,0.4090909,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.272727,0,0,0,0,0.181818,0.272727,0],[0.4090909,0,0,0,0,0.272727,0.4090909,0],[0,0,0,0,0,0,0,0]]

#,ghz,
theo_code_5_1=[[0.8,0,0,0,0,0,0,0.4],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.4,0,0,0,0,0,0,0.2]]
theo_code_5_2=[[0.692307,0,0,0,0,0,0,0.4615384],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.4615384,0,0,0,0,0,0,0.3076923]]
theo_code_5_3=[[0.73529,0,0,0,0,0,0,0.4411764],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.4411764,0,0,0,0,0,0,0.2647058]]
theo_code_5_4=[[0.13793103,0,0,0,0,0,0,0.3448275],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.3448275,0,0,0,0,0,0,0.8620689]]
theo_code_5_5=[[0.662162,0,0,0,0,0,0,0.4729729],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0],[0.4729729,0,0,0,0,0,0,0.337837]]

theo_code_0 = [theo_code_0_1,theo_code_0_2,theo_code_0_3,theo_code_0_4,theo_code_0_5]
theo_code_1 = [theo_code_1_1,theo_code_1_2,theo_code_1_3,theo_code_1_4,theo_code_1_5]
theo_code_2 = [theo_code_2_1,theo_code_2_2,theo_code_2_3,theo_code_2_4,theo_code_2_5]
theo_code_3 = [theo_code_3_1,theo_code_3_2,theo_code_3_3,theo_code_3_4,theo_code_3_5]
theo_code_4 = [theo_code_4_1,theo_code_4_2,theo_code_4_3,theo_code_4_4,theo_code_4_5]
theo_code_5 = [theo_code_5_1,theo_code_5_2,theo_code_5_3,theo_code_5_4,theo_code_5_5]

theo_list_all_states = [theo_code_0,theo_code_1,theo_code_2,theo_code_3,theo_code_4,theo_code_5]


In [5]:
def three_tangle_ASD(psi):
    if psi.shape == (8,):
        t000 = psi[0]
        t111 = psi[7]
        tau_abc = 4*(t000**2)*(t111**2)
    
    elif psi.shape == (8,8):
        t000 = psi[0][0]
        t111 = psi[7][7]
        tau_abc = 4*(t000)*(t111)
    
    else : 
        None 
    return tau_abc      

def tangle_bipartition(rho):
    
    rho = Qobj(rho, dims = [[2,2,2],[2,2,2]])
    rho_reduced = {}
    bi_tangle = {}
    for i in range(3):
        key = 'rho'+str(i)
        rho_reduced[key] = rho.ptrace([i])
    rho_red_list = list(rho_reduced.values())
    bi_tangle['tangle(rho_0_12)'] = 4*np.linalg.det(rho_red_list[0])
    bi_tangle['tangle(rho_1_02)'] = 4*np.linalg.det(rho_red_list[1])
    bi_tangle['tangle(rho_2_01)'] = 4*np.linalg.det(rho_red_list[2])
                                      
    #i_tangle = dict((k, 4*np.linalg.det(v)) for k, v in rho_reduced.items())   

    return bi_tangle

def tangle_pairwise(rho):
    
    '''
     defined for three qubit pure states as given in the article 
     https://arxiv.org/pdf/quant-ph/9907047.pdf
    '''
    rho_red = []
    rho_red_tilda = []
    rho_red_product = []
    rho_red_pro_eig = []
    rho_red_pro_trace = []
    rho_red_tangle = []
    tangle_pair = []
    
    pair_tangle = {}
    j=[1,2,0]
    
    U = tensor(sigmay(),sigmay())
    Ud = U.dag()
    rho = Qobj(rho.data.toarray(), dims = [[2,2,2],[2,2,2]])
    for i in range(3):
        
        rho_red.append(rho.ptrace([i,j[i]])) 
        rho_red_tilda.append(U*(rho_red[i].conj())*Ud)
        rho_red_product.append(rho_red[i]*rho_red_tilda[i])
        evalue, evector = (rho_red_product[i]).eigenstates()
        rho_red_pro_eig.append(evalue)
        #rho_red_pro_trace.append(rho_red_product[i].tr())
        #rho_red_tangle.append(rho_red_pro_trace[i] -2*(rho_red_pro_eig[i][2]*rho_red_pro_eig[i][3]))
        rho_red_pro_eig[i] = sorted(list(rho_red_pro_eig[i]),reverse=True)
        k = rho_red_pro_eig[i]
        tangle_pair.append((max(k[0]-(k[1]+k[2]+k[3]),0))**2)
    
    pair_tangle['tangle(rho_01)'] = tangle_pair[0]
    pair_tangle['tangle(rho_12)'] = tangle_pair[1]
    pair_tangle['tangle(rho_20)'] = tangle_pair[2]
    
    return pair_tangle

def three_tangle(rho):
    three_t = []
    bi_t , pi_t = tangle_bipartition(rho) , tangle_pairwise(rho)
    bi_t_list, pi_t_list = list(bi_t.values()), list(pi_t.values())
    three_t = [bi_t_list[0]-(pi_t_list[0]+pi_t_list[2]), bi_t_list[1]-(pi_t_list[1]+pi_t_list[0]), bi_t_list[2]-(pi_t_list[2]+pi_t_list[1])]   
    three_tangle_min = np.min(three_t)
    return three_tangle_min

def det_rho_abc(M):
    det_list=[]
    for i in range(0,3):
        det_list.append(np.real(linalg.det(M.ptrace([i])))) 
        #det_list = np.real(det_list)              
    return det_list[0], det_list[1], det_list[2]

def class_entropy_3T(M):
    three_t = three_tangle_ASD(M)
    M = Qobj(M, dims=[[2,2,2],[2,2,2]])
    
    '''
    replace with the following rank based classification
    s_a = matrix_rank(M.ptrace([0]), hermitian = True)
    s_b = matrix_rank(M.ptrace([1]), hermitian = True)
    s_c = matrix_rank(M.ptrace([2]), hermitian = True)
    '''
    s_a, s_b, s_c = det_rho_abc(M)
    
    return [s_a, s_b, s_c, three_t]
    #return list_all

# defining functions for checking positive-semi-definiteness 
def PSD(M, tol):
    eigen_val = np.linalg.eigvalsh(M)
    if np.all(eigen_val >= -tol ) == True :
        return True 
    else :
        return False 

def check_valid_rho(M, tol_psd):
    shape = M.shape
    eigen_values = np.linalg.eigvalsh(M)
    psd = PSD(M, tol_psd)
    trace = M.trace()
    herm = np.all(M == M.conj().T)
    
    return shape, eigen_values, psd, trace, herm

def check_pure_rho(M, tol_idem):
    
    rank = np.linalg.matrix_rank(M)
    M_square = np.matmul(M,M)
    idempotent = np.all(abs(M_square - M) <= tol_idem )
    #idempotent = M_square == M
    trace = M_square.trace()
    
    return rank, idempotent, trace

def check_rho(M,tol_psd, tol_idem):
    
    '''
    print(" Shape: {0},\n PSD: {2} ,\n trace: {3} ,\n hermitian: {4} ,\n eigen values:\n {1} ,\n rank: {5},
          \n idempotent: {6} ,\n trace_rho_square: {7}".format(sh, eig, psd, tr, he, rank, idem, tr))
    '''
    sh, eig, psd, tr, he = check_valid_rho(M, tol_psd)
    rank, idem, tr2 =  check_pure_rho(M, tol_idem)
    return sh, eig, psd, tr, he, rank, idem, tr2 


def fs_check(M):
    '''
    # Function to check the full separability of the generated state above 
    # based on the metrics in the paper https://arxiv.org/pdf/quant-ph/0005115.pdf
    '''
    # defining a list to store the ranks 
    rank_list=[]
    M = Qobj(M, dims=[[2,2,2],[2,2,2]])
    for i in range(0,3):
        # calculating the rank of the reduced density matrix of the ith subsystem
        rank_list.append(matrix_rank(M.ptrace([i]),hermitian=True))
    # if ranks of the all the 1 qubit subsystems are equal to 1 then it returns 1
    # meaning that the input Qobj represents a Fully-Separanle three qubit state
    if rank_list[0] == rank_list[1] == rank_list[2] == 1:
        return 1
    else:
        return 0 

def bs1_check(M):
    
    # defining a list to store the ranks 
    rank_list=[]
    M = Qobj(M, dims=[[2,2,2],[2,2,2]])
    for i in range(0,3):
        # calculating the rank of the reduced density matrix of the ith subsystem
        rank_list.append(matrix_rank(M.ptrace([i]),hermitian=True))
        
    # if ranks of the all the 1 qubit subsystems are equal to 1 then it returns 1
    # meaning that the input Qobj represents a Fully-Separanle three qubit state
    if rank_list[1] == rank_list[2] == 2 and rank_list[0] == 1:
        return 1
    else:
        return 0 
    
    
def bs2_check(M):
    rank_list=[]
    M = Qobj(M, dims=[[2,2,2],[2,2,2]])
    for i in range(0,3):
        rank_list.append(matrix_rank(M.ptrace([i]),hermitian=True))
    if rank_list[1] == 1 and rank_list[0] == rank_list[2] == 2:
        return 1
    else:
        return 0 
    
def bs3_check(M):

    rank_list=[]
    M = Qobj(M, dims=[[2,2,2],[2,2,2]])
    for i in range(0,3):
        rank_list.append(matrix_rank(M.ptrace([i]),hermitian=True))
    if rank_list[2] == 1 and rank_list[0] == rank_list[1] == 2:
        return 1
    else:
        return 0       
    
#list_all_states = df_list
def wit_o(rho):
    rho = rho.reshape(8,8)
    O = 2*tensor(sigmax(),sigmax(),sigmax())
    O1 = 2*tensor(sigmax(),sigmax(),sigmaz())
    O2 = 2*tensor(sigmax(),sigmaz(),sigmax())
    O3 = 2*tensor(sigmaz(),sigmax(),sigmax())
    O_exp = np.matmul(rho, O.data.toarray()).trace()
    O1_exp = np.matmul(rho, O1.data.toarray()).trace()
    O2_exp = np.matmul(rho, O2.data.toarray()).trace()
    O3_exp = np.matmul(rho, O3.data.toarray()).trace()
    
    return [O_exp,O1_exp,O2_exp,O3_exp]

def negativity(M):
    eig_list = []
    M = Qobj(M, dims=[[2,2,2],[2,2,2]])
    eig_0, vec_0 = np.linalg.eig(M.ptrace([0]).data.toarray())
    eig_1, vec_1 = np.linalg.eig(M.ptrace([1]).data.toarray())
    eig_2, vec_2 = np.linalg.eig(M.ptrace([2]).data.toarray())
    eig_list.extend(eig_0)
    eig_list.extend(eig_1)
    eig_list.extend(eig_2)
    min_eig = np.min(eig_list)
    return min_eig

def von_neumann_entropy(rho, base, sparse_condition):
    '''
    expects rho in numpy array form, 
    base = base of the logarithm
    sparse condition is either true or false
    '''
    rho = Qobj(rho)
    vne = entropy_vn(rho, base=base,sparse = sparse_condition)
    return vne

def entropy_tangle_classifier(et_list):
    sa =et_list[0]
    sb =et_list[1]
    sc =et_list[2]
    t3 =et_list[3]
    
    if sa == sb ==sc ==t3 ==0 :
        code =0
    elif sa ==t3 ==0 and sb > 0 and sc > 0 :
        code =1 
    elif sb ==t3 ==0 and sa > 0 and sb > 0 :
        code =2 
    elif sc ==t3 ==0 and sa > 0 and sb > 0 :
        code =3 
    elif sa > 0 and sb> 0 and sc > 0 and t3 == 0 :
        code =4 
    elif sa > 0 and sb > 0 and sc > 0 and t3 > 0 :
        code =5 
    else :
        code =None
    return code

In [6]:
wit_op_list = []
for j in range(len(list_all_states)):
    for i in range(len(list_all_states[j])):
        wit_op_list.append(wit_o(np.array(list_all_states[j][i]).reshape(8,8)))

bad_fid =  [0.7646, 0.8913, 0.9612 , 0.9246 ]
bad_id_list = [9, 32, 0, 29]
for bad_id in bad_id_list:
    del wit_op_list[bad_id]
    
wit_op_list = np.real(np.round(wit_op_list,1)).tolist()
#wit_o_list = [i[0] for i in wit_op_list]
#wit_o1_list = [i[1] for i in wit_op_list]
#wit_o2_list = [i[2] for i in wit_op_list]
#wit_o3_list = [i[3] for i in wit_op_list]

# wit_o1_list, wit_o2_list, wit_o3_list
# 'EXPO1', 'EXPO2','EXPO3'


df4 = pd.DataFrame(wit_op_list, columns = ['EXPO','EXPO1', 'EXPO2','EXPO3'])

theo_wit_op_list = []
for j in range(len(theo_list_all_states)):
    for i in range(len(theo_list_all_states[j])):
        theo_wit_op_list.append(wit_o(np.array(theo_list_all_states[j][i]).reshape(8,8)))
        
theo_wit_op_list = np.real(np.round(theo_wit_op_list,1)).tolist()       
theo_wit_o_list = [i[0] for i in theo_wit_op_list]
theo_wit_o1_list = [i[1] for i in theo_wit_op_list]
theo_wit_o2_list = [i[2] for i in theo_wit_op_list]
theo_wit_o3_list = [i[3] for i in theo_wit_op_list]

#df4[['theo_O','1','2','3']] = theo_wit_op_list
df4['theo_O'] = theo_wit_o_list
df4['theo_O1'] = theo_wit_o1_list
df4['theo_O2'] = theo_wit_o2_list
df4['theo_O3'] = theo_wit_o3_list


exp_neg_list = []
for j in range(len(list_all_states)):
    for i in range(len(list_all_states[j])):
        exp_neg_list.append(negativity(np.array(list_all_states[j][i]).reshape(8,8)))

for bad_id in bad_id_list:
    del exp_neg_list[bad_id]

theo_neg_list = []
for j in range(len(theo_list_all_states)):
    for i in range(len(theo_list_all_states[j])):
        theo_neg_list.append(negativity(np.array(theo_list_all_states[j][i]).reshape(8,8)))
        
#for bad_id in bad_id_list:
#    del theo_neg_list[bad_id]        
        

theo_neg_list = np.real(np.round(theo_neg_list,1)).tolist()   
exp_neg_list = np.real(np.round(exp_neg_list,1)).tolist()  

df4['theo_neg'] = theo_neg_list 
df4['exp_neg'] = exp_neg_list 
#df4
        #M = np.round(np.array(list_all_states[eclass][state]).reshape((8,8)),2)
        #M = trunc_rho(np.array(list_all_states[eclass][state]).reshape((8,8)),2)

In [7]:
exp_entropy_tangle_list = []
exp_code = []
for eclass in range(6):
    for state in range(5):
        M = np.array(list_all_states[eclass][state]).reshape((8,8))
        exp_et_list = np.real(np.round(class_entropy_3T(M),1)).tolist()
        exp_entropy_tangle_list.append(exp_et_list)
        exp_code.append(entropy_tangle_classifier(exp_et_list))
        
df1 = pd.DataFrame(exp_entropy_tangle_list, columns= ['exp_Sa','exp_Sb','exp_Sc','exp_3t'])

property_list = []
tol_psd = 1e-10
tol_idem = 1e-10

for eclass in range(6):
    for state in range(5):
        M = np.array(list_all_states[eclass][state]).reshape((8,8))
        sh, eig, psd, tr, he, rank, idem, tr2 = check_rho(M ,tol_psd, tol_idem)
        SEP, BS1,BS2, BS3 = fs_check(M), bs1_check(M), bs2_check(M), bs3_check(M)
        property_list.append([psd, tr, he, rank, idem, tr2, SEP, BS1,BS2, BS3 ])
        #property_list.append(np.real(np.round([psd, tr, he, rank, idem, tr2, SEP, BS1,BS2, BS3 ],1)).tolist())
        
df2 = pd.DataFrame(property_list, columns= ['PSD','TRACE','HERM','RANK','IDEM','TRACE^2','SEP','BS1','BS2','BS3'])
df3 = pd.concat([df1,df2], axis = 1)
df3
#df3[['CODE', 'FIDELITY', 'S_a', 'Sb', 'Sc', '3t']]

,exp_Sa,exp_Sb,exp_Sc,exp_3t,PSD,TRACE,HERM,RANK,IDEM,TRACE^2,SEP,BS1,BS2,BS3
0,0.0,0.0,0.0,0.0,False,1.0000+0.0000j,True,8,False,0.954012-0.000000j,0,0,0,0
1,0.0,0.1,0.0,0.0,False,1.0000+0.0000j,True,8,False,0.819386+0.000000j,0,0,0,0
2,0.0,0.1,0.0,0.0,False,1.0001+0.0000j,True,8,False,0.793171-0.000000j,0,0,0,0
3,0.0,0.1,0.0,0.0,False,1.0000+0.0000j,True,8,False,0.758907+0.000000j,0,0,0,0
4,0.0,0.0,0.0,0.0,False,1.0000+0.0000j,True,8,False,0.856012+0.000000j,0,0,0,0
5,0.0,0.0,0.1,0.0,False,1.0001+0.0000j,True,8,False,0.939591+0.000000j,0,0,0,0
6,0.0,0.1,0.1,0.0,False,0.9999+0.0000j,True,8,False,0.913093-0.000000j,0,0,0,0
7,0.0,0.0,0.0,0.0,False,1.0001+0.0000j,True,8,False,0.937230+0.000000j,0,0,0,0
8,0.2,0.0,0.2,0.0,False,1.0000+0.0000j,True,8,False,0.859138-0.000000j,0,0,0,0
9,0.0,0.2,0.2,0.0,False,1.0000+0.0000j,True,8,False,0.870916+0.000000j,0,0,0,0


In [ ]:
from sympy import Matrix

In [20]:
#Matrix(trunc_rho(np.array(theo_list_all_states[3][0]).reshape((8,8)),6))
#trunc_rho(np.array(theo_list_all_states[3][0]).reshape((8,8)),6)
        #M = np.round(np.array(theo_list_all_states[eclass][state]).reshape((8,8)),2)
        #M = trunc_rho(np.array(theo_list_all_states[eclass][state]).reshape((8,8)),2)
        #exp_entropy_tangle_list.append(class_entropy_3T(M))

In [8]:
theo_entropy_tangle_list = []
theo_code = []
for eclass in range(6):
    for state in range(5):
        M = np.array(theo_list_all_states[eclass][state]).reshape((8,8))
        theo_et_list = np.real(np.round(class_entropy_3T(M),1)).tolist()
        theo_entropy_tangle_list.append(theo_et_list)
        theo_code.append(entropy_tangle_classifier(theo_et_list))
        
theo_df1 = pd.DataFrame(theo_entropy_tangle_list, columns= ['theo_Sa','theo_Sb','theo_Sc','theo_3t'])
        
theo_property_list = []
tol_psd = 1e-10
tol_idem = 1e-10

for eclass in range(6):
    for state in range(5):
        M = np.array(theo_list_all_states[eclass][state]).reshape((8,8))
        sh, eig, psd, tr, he, rank, idem, tr2 = check_rho(M ,tol_psd, tol_idem)
        SEP, BS1,BS2, BS3 = fs_check(M), bs1_check(M), bs2_check(M), bs3_check(M)
        theo_property_list.append([psd, tr, he, rank, idem, tr2, SEP, BS1,BS2, BS3 ])
        #property_list.append(np.real(np.round([psd, tr, he, rank, idem, tr2, SEP, BS1,BS2, BS3 ],1)).tolist())
        
theo_df2 = pd.DataFrame(theo_property_list, columns= ['PSD','TRACE','HERM','RANK','IDEM','TRACE^2','SEP','BS1','BS2','BS3'])

theo_df3 = pd.concat([theo_df1,theo_df2], axis = 1)
theo_df3

,theo_Sa,theo_Sb,theo_Sc,theo_3t,PSD,TRACE,HERM,RANK,IDEM,TRACE^2,SEP,BS1,BS2,BS3
0,0.0,0.0,-0.0,0.0,False,1.000000,True,2,False,0.999999,0,0,0,0
1,0.0,0.0,0.0,0.0,True,1.000000,True,1,True,1.000000,1,0,0,0
2,0.0,0.0,0.0,0.0,True,0.999996,True,2,False,0.999983,0,0,0,0
3,0.0,0.0,0.0,0.0,True,1.000000,True,2,False,0.999999,0,0,0,0
4,0.0,0.0,0.0,0.0,True,1.000000,True,1,True,1.000000,1,0,0,0
5,0.0,0.1,0.1,0.0,True,1.000000,True,1,False,0.999999,0,1,0,0
6,0.0,0.0,0.0,0.0,False,1.000000,True,2,False,1.000000,0,1,0,0
7,0.0,0.1,0.1,0.0,False,1.000000,True,2,False,1.000001,0,1,0,0
8,0.0,0.1,0.1,0.0,False,0.999998,True,2,False,0.999997,0,1,0,0
9,0.0,0.0,0.0,0.0,True,0.999999,True,1,False,0.999998,0,1,0,0


In [9]:
df_list = [df_0, df_1 , df_2, df_3, df_4, df_5]
data_expt = pd.concat(df_list)
#data_expt_red = dim_reduce(data_expt)
data_expt.reset_index(inplace = True)
new_df = pd.concat([data_expt[['CODE', 'FIDELITY']], df1, theo_df1], axis = 1)
#new_df[['CODE', 'FIDELITY', 'theo_Sa','exp_Sa','theo_Sb','exp_Sb', 'theo_Sc','exp_Sc', 'theo_3t','exp_3t'  ]] 
new_df = new_df[['theo_Sa','exp_Sa','theo_Sb','exp_Sb', 'theo_Sc','exp_Sc']] 
#new_df.replace(0.00,0)
new_df_string = new_df.astype(str)
new_df_latex = new_df_string.stack().groupby(level=0).apply(' & '.join)
type(new_df_latex)
new_df_latex

0     0.0 & 0.0 & 0.0 & 0.0 & -0.0 & 0.0
1      0.0 & 0.0 & 0.0 & 0.1 & 0.0 & 0.0
2      0.0 & 0.0 & 0.0 & 0.1 & 0.0 & 0.0
3      0.0 & 0.0 & 0.0 & 0.1 & 0.0 & 0.0
4      0.0 & 0.0 & 0.0 & 0.0 & 0.0 & 0.0
5      0.0 & 0.0 & 0.1 & 0.0 & 0.1 & 0.1
6      0.0 & 0.0 & 0.0 & 0.1 & 0.0 & 0.1
7      0.0 & 0.0 & 0.1 & 0.0 & 0.1 & 0.0
8      0.0 & 0.2 & 0.1 & 0.0 & 0.1 & 0.2
9      0.0 & 0.0 & 0.0 & 0.2 & 0.0 & 0.2
10     0.2 & 0.2 & 0.0 & 0.0 & 0.2 & 0.2
11     0.2 & 0.2 & 0.0 & 0.0 & 0.2 & 0.2
12     0.2 & 0.1 & 0.0 & 0.2 & 0.2 & 0.2
13     0.2 & 0.2 & 0.0 & 0.1 & 0.2 & 0.2
14     0.2 & 0.2 & 0.0 & 0.0 & 0.2 & 0.2
15     0.2 & 0.1 & 0.2 & 0.1 & 0.0 & 0.0
16     0.2 & 0.2 & 0.2 & 0.2 & 0.0 & 0.0
17     0.2 & 0.2 & 0.2 & 0.2 & 0.0 & 0.0
18     0.2 & 0.2 & 0.2 & 0.2 & 0.0 & 0.0
19     0.2 & 0.2 & 0.2 & 0.2 & 0.0 & 0.0
20     0.1 & 0.1 & 0.2 & 0.1 & 0.2 & 0.1
21     0.2 & 0.1 & 0.2 & 0.2 & 0.1 & 0.2
22     0.2 & 0.1 & 0.1 & 0.2 & 0.1 & 0.1
23     0.1 & 0.2 & 0.2 & 0.1 & 0.2 & 0.1
24     0.2 & 0.1

In [10]:
new_df = pd.concat([data_expt[['CODE', 'FIDELITY']], df1, theo_df1], axis = 1)
tangle_df = new_df[['theo_3t','exp_3t']] 
tangle_df_string = tangle_df.astype(str)
tangle_df_latex = tangle_df_string.stack().groupby(level=0).apply(' & '.join)
tangle_df_latex

0     0.0 & 0.0
1     0.0 & 0.0
2     0.0 & 0.0
3     0.0 & 0.0
4     0.0 & 0.0
5     0.0 & 0.0
6     0.0 & 0.0
7     0.0 & 0.0
8     0.0 & 0.0
9     0.0 & 0.0
10    0.0 & 0.0
11    0.0 & 0.0
12    0.0 & 0.1
13    0.0 & 0.0
14    0.0 & 0.0
15    0.0 & 0.0
16    0.0 & 0.0
17    0.0 & 0.0
18    0.0 & 0.0
19    0.0 & 0.0
20    0.0 & 0.0
21    0.0 & 0.0
22    0.0 & 0.0
23    0.0 & 0.0
24    0.0 & 0.0
25    0.6 & 0.3
26    0.9 & 0.6
27    0.8 & 0.5
28    0.5 & 0.4
29    0.9 & 0.3
dtype: object

In [ ]:
def truncate(values, decs):
    return np.trunc(values*10**decs)/(10**decs)

def trunc_rho(rho,dig):
    rho_list = list(rho.reshape((64,)))
    rho_real_list = [truncate(np.real(i),dig) for i in rho_list]
    rho_imag_list = [truncate(np.imag(i),dig) for i in rho_list]
    rho_merge_list = np.array([rho_real_list[i]+ 1j*rho_imag_list[i] for i in range(64)]).reshape(8,8)
    return rho_merge_list